# Tarea 2 - Pregunta 3
## Limpieza y transformación de datos

### Objetivo

En este notebook se realizará el proceso de limpieza y transformación
de los datos de contratación correspondientes a la entidad INVIAS,
tomando como punto de partida los resultados obtenidos en la auditoría
inicial.

El proceso busca preparar los datos para las etapas posteriores de
análisis estadístico y generación de resultados, conservando las
observaciones que puedan representar señales de alerta para la pregunta
de negocio.

### Criterios generales de limpieza

Las transformaciones realizadas en este notebook estarán orientadas a:

- Corregir formatos y tipos de datos.
- Estandarizar variables categóricas y de texto cuando sea necesario.
- Tratar los valores faltantes de acuerdo con el significado de cada
  variable.
- Identificar y tratar registros duplicados cuando corresponda.
- Preparar las variables financieras y temporales para el análisis.
- Conservar los casos identificados durante la auditoría que puedan ser
  relevantes para el análisis de riesgo contractual.

Las decisiones de eliminación o transformación de registros serán
documentadas y justificadas para garantizar la trazabilidad del proceso.

In [112]:
import pandas as pd
import numpy as np

# Cargar el archivo de datos de INVIAS
df = pd.read_csv("../../invias.csv")

# Verificar las dimensiones iniciales del conjunto de datos
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

Número de filas: 25605
Número de columnas: 89


/var/folders/bd/ygh4x6gj0jd4vfxdxdjck_s00000gn/T/ipykernel_2198/2333507425.py:5: DtypeWarning: Columns (0: direccion_de_ejecucion_del_contrato) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../invias.csv")


In [113]:
# Revisar los tipos de datos presentes en la columna problemática

df["direccion_de_ejecucion_del_contrato"].map(type).value_counts()

direccion_de_ejecucion_del_contrato
<class 'float'>    25602
<class 'str'>          3
Name: count, dtype: int64

In [114]:
# Identificar los registros que contienen texto en la columna direccion_de_ejecucion_del_contrato

registros_texto = df[
    df["direccion_de_ejecucion_del_contrato"].apply(lambda x: isinstance(x, str))
]

registros_texto[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "direccion_de_ejecucion_del_contrato"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,direccion_de_ejecucion_del_contrato
10409,CO1.PCCNTR.7898538,1151-2025,terminado,No definido
24503,CO1.PCCNTR.7894143,1147-2025,Cerrado,No definido
24550,CO1.PCCNTR.7898079,1152-2025,terminado,No definido


### 2.1 Tratamiento de `direccion_de_ejecucion_del_contrato`

Durante la carga inicial se identificó una advertencia de tipos mixtos en
la variable `direccion_de_ejecucion_del_contrato`.

La revisión mostró que 25.602 registros corresponden a valores faltantes
(`NaN`) y solamente 3 registros contienen texto. Los tres registros
presentan el valor `"No definido"`.

Dado que la variable no proporciona información efectiva sobre la
dirección de ejecución del contrato y presenta un nivel de ausencia de
información prácticamente total, se decide excluirla del conjunto de
variables utilizado para el análisis.

Esta decisión no implica eliminar registros, sino únicamente retirar una
variable que no aporta información analítica suficiente.

In [115]:
# Eliminar la variable sin información analítica suficiente

df = df.drop(columns=["direccion_de_ejecucion_del_contrato"])

print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

Número de filas: 25605
Número de columnas: 88


### 2.2 Conversión de variables de fecha

Las variables `fecha_de_firma`, `fecha_de_inicio_del_contrato` y
`fecha_de_fin_del_contrato` contienen información temporal relevante
para el análisis contractual.

Estas variables serán convertidas al formato datetime de pandas para
facilitar posteriormente el análisis por año, duración y relaciones
temporales entre los contratos.

Los valores faltantes serán conservados como valores nulos y no serán
imputados en esta etapa, dado que la ausencia de una fecha puede tener
significado para determinados estados contractuales.

In [116]:
# Convertir las variables de fecha al formato datetime

columnas_fecha = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato"
]

for columna in columnas_fecha:
    df[columna] = pd.to_datetime(df[columna], errors="coerce")

# Verificar los tipos de datos resultantes
df[columnas_fecha].dtypes

fecha_de_firma                  datetime64[us]
fecha_de_inicio_del_contrato    datetime64[us]
fecha_de_fin_del_contrato       datetime64[us]
dtype: object

### 2.3 Validación de las variables de fecha

Después de convertir las variables temporales al formato datetime, se
verifica que la transformación haya conservado la cantidad de valores
válidos y faltantes identificados durante la auditoría inicial.

In [117]:
# Verificar valores válidos y faltantes después de la conversión

for columna in columnas_fecha:
    valores_validos = df[columna].notna().sum()
    valores_faltantes = df[columna].isna().sum()

    print(f"{columna}:")
    print(f"  Valores válidos: {valores_validos}")
    print(f"  Valores faltantes: {valores_faltantes}")
    print()

fecha_de_firma:
  Valores válidos: 17477
  Valores faltantes: 8128

fecha_de_inicio_del_contrato:
  Valores válidos: 17795
  Valores faltantes: 7810

fecha_de_fin_del_contrato:
  Valores válidos: 20074
  Valores faltantes: 5531



### 2.4 Validación de coherencia temporal

Una vez convertidas las variables temporales al formato datetime, se
verifica la consistencia cronológica de las fechas contractuales.

Se revisará si existen registros en los cuales:

- La fecha de inicio sea anterior a la fecha de firma.
- La fecha de finalización sea anterior a la fecha de inicio.

Los registros con fechas faltantes no serán considerados inconsistentes
en esta validación, dado que no existe información suficiente para
establecer una relación cronológica.

In [118]:
# Validar relaciones cronológicas entre las fechas del contrato

inicio_antes_firma = df[
    df["fecha_de_inicio_del_contrato"].notna() &
    df["fecha_de_firma"].notna() &
    (df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"])
]

fin_antes_inicio = df[
    df["fecha_de_fin_del_contrato"].notna() &
    df["fecha_de_inicio_del_contrato"].notna() &
    (df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"])
]

print("Inicio anterior a firma:", len(inicio_antes_firma))
print("Fin anterior a inicio:", len(fin_antes_inicio))

Inicio anterior a firma: 2959
Fin anterior a inicio: 1


### 2.5 Investigación de fechas de inicio anteriores a la firma

La validación temporal identificó 2.959 contratos cuya fecha de inicio
registrada es anterior a la fecha de firma.

Debido al número significativo de casos, no se asumirán automáticamente
como errores de calidad de datos. Se realizará una revisión descriptiva
por estado del contrato para determinar si el comportamiento se concentra
en determinados estados contractuales.

Los registros serán conservados mientras se determina su tratamiento.

In [119]:
# Distribución por estado de los contratos con inicio anterior a la firma

inicio_antes_firma["estado_contrato"].value_counts()

estado_contrato
Cerrado       1156
terminado      826
Modificado     504
Aprobado       470
Suspendido       3
Name: count, dtype: int64

In [120]:
# Porcentaje de casos con inicio anterior a la firma sobre los contratos
# que tienen disponibles ambas fechas

total_con_firma_inicio = (
    df["fecha_de_firma"].notna() &
    df["fecha_de_inicio_del_contrato"].notna()
).sum()

porcentaje_inicio_antes_firma = (
    len(inicio_antes_firma) / total_con_firma_inicio * 100
)

print("Contratos con firma e inicio disponibles:", total_con_firma_inicio)
print("Inicio anterior a firma:", len(inicio_antes_firma))
print(f"Porcentaje: {porcentaje_inicio_antes_firma:.2f}%")

Contratos con firma e inicio disponibles: 17285
Inicio anterior a firma: 2959
Porcentaje: 17.12%


In [121]:
fin_antes_inicio[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "fecha_de_firma",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato
13181,CO1.PCCNTR.8746765,2320-2025,En ejecución,2025-12-31,2026-01-05,2025-09-04


### 2.6 Creación de indicadores de inconsistencias temporales

La revisión de las fechas identificó 2.959 contratos cuya fecha de inicio
es anterior a la fecha de firma, equivalentes al 17,12 % de los contratos
con ambas fechas disponibles.

Debido a la concentración de estos casos en determinados estados
contractuales, no se consideran automáticamente errores de digitación y
se conservan las fechas originales.

Adicionalmente, se identificó un contrato cuya fecha de finalización es
anterior a la fecha de inicio. Este caso será igualmente conservado.

Para preservar la información original y facilitar el análisis posterior,
se crean variables indicadoras que permitan identificar estas
inconsistencias sin modificar las fechas de origen.

In [122]:
# Crear indicadores de inconsistencias temporales

df["inicio_antes_firma"] = (
    df["fecha_de_inicio_del_contrato"].notna() &
    df["fecha_de_firma"].notna() &
    (df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"])
)

df["fin_antes_inicio"] = (
    df["fecha_de_fin_del_contrato"].notna() &
    df["fecha_de_inicio_del_contrato"].notna() &
    (df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"])
)

print("Inicio anterior a firma:", df["inicio_antes_firma"].sum())
print("Fin anterior a inicio:", df["fin_antes_inicio"].sum())

Inicio anterior a firma: 2959
Fin anterior a inicio: 1


### 2.7 Revisión de valores faltantes

Después de las transformaciones realizadas, se revisa nuevamente la
completitud de las variables.

Los valores faltantes no serán eliminados de manera general, dado que
pueden representar situaciones diferentes dependiendo de la variable y
del contexto contractual.

La decisión sobre su tratamiento se realizará variable por variable,
considerando su relevancia para el análisis y el porcentaje de
información disponible.

In [123]:
# Revisar valores faltantes después de las transformaciones

faltantes = pd.DataFrame({
    "no_nulos": df.notna().sum(),
    "faltantes": df.isna().sum(),
    "porcentaje_faltantes": df.isna().mean() * 100
})

faltantes = faltantes.sort_values(
    "porcentaje_faltantes",
    ascending=False
)

faltantes

,no_nulos,faltantes,porcentaje_faltantes
fecha_de_notificacion_de_prorrogacion,2726,22879,89.353642
fecha_fin_liquidacion,6036,19569,76.426479
fecha_inicio_liquidacion,6036,19569,76.426479
ultima_actualizacion,14553,11052,43.163445
fecha_de_firma,17477,8128,31.743800
...,...,...,...
proceso_de_compra,25605,0,0.000000
id_contrato,25605,0,0.000000
referencia_del_contrato,25605,0,0.000000
estado_contrato,25605,0,0.000000


### 2.8 Revisión de variables financieras

Debido al enfoque financiero de la Pregunta 3, se revisan de manera
particular las variables asociadas con el valor y la ejecución económica
de los contratos.

El análisis se concentra en las variables:

- `valor_del_contrato`
- `valor_facturado`
- `valor_pagado`
- `valor_pendiente_de_pago`
- `valor_pendiente_de_ejecucion`
- `valor_amortizado`
- `saldo_cdp`
- `saldo_vigencia`

Estas variables fueron previamente identificadas como disponibles para
el subconjunto financiero de 20.718 registros.

Se verificará nuevamente su completitud antes de continuar con las
transformaciones.

In [124]:
# Revisar completitud de las variables financieras

variables_financieras = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia"
]

faltantes_financieros = pd.DataFrame({
    "no_nulos": df[variables_financieras].notna().sum(),
    "faltantes": df[variables_financieras].isna().sum(),
    "porcentaje_faltantes": (
        df[variables_financieras].isna().mean() * 100
    )
})

faltantes_financieros

,no_nulos,faltantes,porcentaje_faltantes
valor_del_contrato,20718,4887,19.086116
valor_facturado,20718,4887,19.086116
valor_pagado,20718,4887,19.086116
valor_pendiente_de_pago,20718,4887,19.086116
valor_pendiente_de_ejecucion,20718,4887,19.086116
valor_amortizado,20718,4887,19.086116
saldo_cdp,20718,4887,19.086116
saldo_vigencia,20718,4887,19.086116


### 2.9 Construcción del subconjunto financiero

La revisión de completitud mostró que las ocho variables financieras
analizadas presentan exactamente el mismo patrón de faltantes: 20.718
registros con información y 4.887 registros sin información financiera.

Debido a que estos faltantes se presentan de manera conjunta, se
construye un subconjunto financiero a partir de los contratos que cuentan
con `valor_del_contrato`.

El conjunto de datos principal conserva los 25.605 contratos originales,
mientras que `df_financiero` contiene únicamente los 20.718 contratos con
información financiera disponible.

Los 4.887 registros sin información financiera no se eliminan del
conjunto principal, ya que podrían ser relevantes para otros análisis.

In [125]:
# Crear subconjunto de contratos con información financiera

df_financiero = df[
    df["valor_del_contrato"].notna()
].copy()

print("Registros del conjunto principal:", len(df))
print("Registros del subconjunto financiero:", len(df_financiero))

Registros del conjunto principal: 25605
Registros del subconjunto financiero: 20718


### 2.10 Revisión de variables categóricas

Se revisan las variables categóricas del conjunto de datos para
identificar valores inconsistentes, marcadores de información ausente,
errores de estandarización o valores atípicos en su representación.

No se modificarán los valores en esta etapa. Primero se identificarán
los valores con baja frecuencia y aquellos que puedan representar
información faltante o registros no estandarizados.

In [126]:
# Identificar las variables categóricas del conjunto de datos

columnas_categoricas = df.select_dtypes(
    include=["object", "string"]
).columns

print("Número de variables categóricas:", len(columnas_categoricas))
print("\nVariables categóricas:")
print(columnas_categoricas.tolist())

Número de variables categóricas: 65

Variables categóricas:
['id', 'version', 'created_at', 'updated_at', 'nombre_entidad', 'departamento', 'ciudad', 'localizacion', 'orden', 'sector', 'rama', 'entidad_centralizada', 'proceso_de_compra', 'id_contrato', 'referencia_del_contrato', 'estado_contrato', 'codigo_de_categoria_principal', 'descripcion_del_proceso', 'tipo_de_contrato', 'modalidad_de_contratacion', 'justificacion_modalidad_de', 'condiciones_de_entrega', 'tipodocproveedor', 'documento_proveedor', 'proveedor_adjudicado', 'es_grupo', 'es_pyme', 'habilita_pago_adelantado', 'liquidacion', 'obligacion_ambiental', 'obligaciones_postconsumo', 'reversion', 'origen_de_los_recursos', 'destino_gasto', 'espostconflicto', 'puntos_del_acuerdo', 'pilares_del_acuerdo', 'urlproceso', 'nombre_representante_legal', 'nacionalidad_representante_legal', 'domicilio_representante_legal', 'tipo_de_identificaci_n_representante_legal', 'identificaci_n_representante_legal', 'genero_representante_legal', 'ult

### 2.11 Revisión de categorías relevantes

Se revisan inicialmente las variables categóricas con mayor relevancia
para la caracterización de los contratos: estado, tipo y modalidad de
contratación, ubicación geográfica y proveedor adjudicado.

El objetivo es identificar categorías inconsistentes, valores que
representen ausencia de información y diferencias de estandarización.

En esta etapa no se modifican los valores originales.

In [127]:
# Revisar las categorías de las variables seleccionadas

variables_categoricas_relevantes = [
    "estado_contrato",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "departamento",
    "proveedor_adjudicado"
]

for columna in variables_categoricas_relevantes:
    print(f"\n--- {columna} ---")
    print("Número de categorías:", df[columna].nunique(dropna=False))
    print(df[columna].value_counts(dropna=False).head(15))


--- estado_contrato ---
Número de categorías: 11
estado_contrato
Cerrado              9263
Modificado           5434
terminado            3416
En ejecución         2512
Borrador             2438
Aprobado              872
Cancelado             629
Suspendido            361
enviado Proveedor     301
En aprobación         287
cedido                 92
Name: count, dtype: int64

--- tipo_de_contrato ---
Número de categorías: 16
tipo_de_contrato
Prestación de servicios       12058
NaN                            4887
Obra                           3409
Otro                           2326
Interventoría                  1530
Suministros                     593
Consultoría                     573
Compraventa                      81
Comodato                         74
Seguros                          32
Arrendamiento de inmuebles       31
Acuerdo Marco de Precios          5
Concesión                         2
Decreto 092 de 2017               2
Arrendamiento de muebles          1
Name: count, d

In [128]:
# Revisar variabilidad de la ubicación geográfica

print("Ciudades:")
print(df["ciudad"].value_counts(dropna=False))

print("\nNúmero de ciudades:", df["ciudad"].nunique(dropna=False))

Ciudades:
ciudad
Bogotá    25605
Name: count, dtype: int64

Número de ciudades: 1


### 2.12 Eliminación de variable sin variabilidad

La variable `departamento` presenta una única categoría para la totalidad
de los 25.605 registros: `Distrito Capital de Bogotá`.

Al no presentar variabilidad, esta variable no aporta información
discriminante para el análisis y se elimina del conjunto de datos.

In [129]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["departamento"])

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Número de filas: 25605
Número de columnas: 89


In [130]:
print("departamento presente:", "departamento" in df.columns)

departamento presente: False


### 2.13 Revisión de valores especiales en la referencia del contrato

Se revisan valores de baja frecuencia presentes en `referencia_del_contrato`
para identificar posibles marcadores de ausencia de información o valores
no estandarizados.

No se modificarán los registros hasta determinar el tratamiento adecuado.

In [131]:
# Revisar valores especiales en referencia del contrato

valores_especiales = [".", "-", "0", "XXXX", "CANCELADO"]

for valor in valores_especiales:
    print(f"\nValor: {valor}")
    print("Cantidad:", (df["referencia_del_contrato"] == valor).sum())


Valor: .
Cantidad: 12

Valor: -
Cantidad: 6

Valor: 0
Cantidad: 5

Valor: XXXX
Cantidad: 4

Valor: CANCELADO
Cantidad: 7


In [132]:
# Revisar el contexto de los valores especiales

df[df["referencia_del_contrato"].isin(valores_especiales)][[
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",
    "proveedor_adjudicado"
]].sort_values("referencia_del_contrato")

,id_contrato,referencia_del_contrato,estado_contrato,proveedor_adjudicado
2230,CO1.PCCNTR.3175096,-,Borrador,JUAN CARLOS VANEGAS MUÑOZ
5660,CO1.PCCNTR.3174995,-,Borrador,CARLOS JULIO ROMERO ANTURY
9167,CO1.PCCNTR.3182314,-,Borrador,JOSE DAVID PADILLA ROMERO
9280,CO1.PCCNTR.9258475,-,Borrador,NaN
11690,CO1.PCCNTR.3174994,-,Borrador,DANIEL GEOVANNY FONSECA ZAMBRANO
25441,CO1.PCCNTR.3175173,-,Borrador,Julio Andres Ulloa Palomo
52,CO1.PCCNTR.7398046,.,Cancelado,Sin Descripcion
2417,CO1.PCCNTR.7335390,.,Borrador,NaN
4247,CO1.PCCNTR.8974804,.,Borrador,NaN
4684,CO1.PCCNTR.8877841,.,Borrador,NaN


#### Tratamiento de valores especiales

La revisión contextual permitió identificar dos grupos de valores
especiales.

Los valores `.`, `-`, `0` y `XXXX` se consideran valores no informativos
para la variable `referencia_del_contrato`, debido a su baja frecuencia y
a que aparecen principalmente en contratos en estado Borrador o Cancelado.

Estos valores serán reemplazados por valores nulos (`NaN`), preservando la
distinción entre ausencia de información y una referencia contractual
válida.

El valor `CANCELADO` se conserva, debido a que se encuentra asociado a
registros cuyo estado contractual puede ser Cancelado y no existe
evidencia suficiente para considerarlo un error de calidad.

In [133]:
# Reemplazar valores no informativos en la referencia del contrato

valores_no_informativos = [".", "-", "0", "XXXX"]

df["referencia_del_contrato"] = df["referencia_del_contrato"].replace(
    valores_no_informativos,
    pd.NA
)

In [134]:
# Validar el tratamiento de valores especiales

for valor in valores_no_informativos:
    print(
        f"{valor}:",
        (df["referencia_del_contrato"] == valor).sum()
    )

print(
    "\nCANCELADO:",
    (df["referencia_del_contrato"] == "CANCELADO").sum()
)

print(
    "\nValores faltantes en referencia:",
    df["referencia_del_contrato"].isna().sum()
)

.: 0
-: 0
0: 0
XXXX: 0

CANCELADO: 7

Valores faltantes en referencia: 27


In [135]:
print("Valores faltantes en referencia:", 
      df["referencia_del_contrato"].isna().sum())

print("Total de registros:", len(df))

print("Referencias no faltantes:", 
      df["referencia_del_contrato"].notna().sum())

Valores faltantes en referencia: 27
Total de registros: 25605
Referencias no faltantes: 25578


In [136]:
print(df["referencia_del_contrato"].value_counts(dropna=False).head(10))

referencia_del_contrato
NaN             27
CANCELADO        7
3655-2023        6
0079-2022        4
4616-2023        4
4239-2023        4
0385-2025        4
4528-2023        4
1941 DE 2024     4
4134-2023        4
Name: count, dtype: int64


Se identificaron 27 valores no informativos (`.`, `-`, `0` y `XXXX`) en `referencia_del_contrato`, los cuales fueron transformados a valores faltantes. El valor `CANCELADO`, presente en 7 registros, fue conservado debido a que corresponde a una categoría observada en el contexto contractual y no se encontró evidencia suficiente para considerarlo un error.

### 2.14 Revisión de valores no informativos en proveedor adjudicado

La variable `proveedor_adjudicado` presenta 4.887 valores faltantes y
791 registros con el texto `Sin Descripcion`.

Se revisará el contexto de estos registros para determinar si `Sin
Descripcion` representa ausencia de información y debe homologarse con
los valores faltantes.

In [137]:
# Revisar el contexto de "Sin Descripcion"

df[df["proveedor_adjudicado"] == "Sin Descripcion"][[
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",
    "proveedor_adjudicado",
    "valor_del_contrato"
]].head(20)

,id_contrato,referencia_del_contrato,estado_contrato,proveedor_adjudicado,valor_del_contrato
4,CO1.PCCNTR.3516681,CO1.PCCNTR.3516681,Cancelado,Sin Descripcion,0.0
18,CO1.PCCNTR.6838297,CO1.PCCNTR.6838297,Cancelado,Sin Descripcion,0.0
51,CO1.PCCNTR.8277719,CO1.PCCNTR.8277719,Borrador,Sin Descripcion,0.0
52,CO1.PCCNTR.7398046,NaN,Cancelado,Sin Descripcion,0.0
59,CO1.PCCNTR.7766838,CO1.PCCNTR.7766838,Cancelado,Sin Descripcion,0.0
84,CO1.PCCNTR.8813998,CO1.PCCNTR.8813998,Borrador,Sin Descripcion,0.0
120,CO1.PCCNTR.5353027,CO1.PCCNTR.5353027,Cancelado,Sin Descripcion,0.0
142,CO1.PCCNTR.344809,CO1.PCCNTR.344809,Borrador,Sin Descripcion,0.0
157,CO1.PCCNTR.5035123,CO1.PCCNTR.5035123,Borrador,Sin Descripcion,0.0
195,CO1.PCCNTR.484706,CO1.PCCNTR.484706,Cancelado,Sin Descripcion,0.0


In [138]:
# Contexto de los registros con proveedor sin descripción

sin_descripcion = df[df["proveedor_adjudicado"] == "Sin Descripcion"]

print("Total 'Sin Descripcion':", len(sin_descripcion))

print("\nEstados del contrato:")
print(sin_descripcion["estado_contrato"].value_counts())

print("\nValor del contrato:")
print(sin_descripcion["valor_del_contrato"].describe())

print("\nCon valor del contrato igual a 0:")
print(
    (sin_descripcion["valor_del_contrato"] == 0).sum()
)

print("\nCon referencia del contrato faltante:")
print(
    sin_descripcion["referencia_del_contrato"].isna().sum()
)

Total 'Sin Descripcion': 791

Estados del contrato:
estado_contrato
Borrador     524
Cancelado    267
Name: count, dtype: int64

Valor del contrato:
count    7.910000e+02
mean     9.190316e+11
std      1.867711e+13
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.410000e+14
Name: valor_del_contrato, dtype: float64

Con valor del contrato igual a 0:
774

Con referencia del contrato faltante:
2


#### Tratamiento de "Sin Descripcion"

Se identificaron 791 registros con el valor `Sin Descripcion` en la variable
`proveedor_adjudicado`.

Los registros corresponden exclusivamente a contratos en estado `Borrador`
(524) o `Cancelado` (267). Adicionalmente, 774 de los 791 registros presentan
un valor del contrato igual a cero.

Debido a este patrón, se considera que `Sin Descripcion` representa ausencia
de información sobre el proveedor adjudicado y se homologa con un valor
faltante (`NaN`).

Esta transformación no implica afirmar que los registros sean inválidos,
sino que se estandariza la representación de ausencia de información para
facilitar el análisis posterior.

In [139]:
# Homologar "Sin Descripcion" como valor faltante

df["proveedor_adjudicado"] = df["proveedor_adjudicado"].replace(
    "Sin Descripcion",
    pd.NA
)

In [140]:
# Validar la transformación

print(
    "Registros con 'Sin Descripcion':",
    (df["proveedor_adjudicado"] == "Sin Descripcion").sum()
)

print(
    "Valores faltantes en proveedor_adjudicado:",
    df["proveedor_adjudicado"].isna().sum()
)

Registros con 'Sin Descripcion': 0
Valores faltantes en proveedor_adjudicado: 5678


In [141]:
# Revisar valores faltantes después de las transformaciones

faltantes = pd.DataFrame({
    "no_nulos": df.notna().sum(),
    "faltantes": df.isna().sum(),
})

faltantes["porcentaje_faltantes"] = (
    faltantes["faltantes"] / len(df) * 100
)

faltantes.sort_values(
    "porcentaje_faltantes",
    ascending=False
).head(20)

,no_nulos,faltantes,porcentaje_faltantes
fecha_de_notificacion_de_prorrogacion,2726,22879,89.353642
fecha_inicio_liquidacion,6036,19569,76.426479
fecha_fin_liquidacion,6036,19569,76.426479
ultima_actualizacion,14553,11052,43.163445
fecha_de_firma,17477,8128,31.743800
fecha_de_inicio_del_contrato,17795,7810,30.501855
proveedor_adjudicado,19927,5678,22.175356
fecha_de_fin_del_contrato,20074,5531,21.601250
nombre_ordenador_del_gasto,20699,4906,19.160320
numero_de_cuenta,20699,4906,19.160320


In [142]:
# Identificar el patrón común de faltantes

columnas_4906 = [
    "nombre_ordenador_del_gasto",
    "numero_de_cuenta",
    "tipo_de_cuenta",
    "el_contrato_puede_ser_prorrogado",
    "objeto_del_contrato",
    "nombre_del_banco"
]

df[df[columnas_4906].isna().all(axis=1)][[
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",
    "valor_del_contrato",
    "proveedor_adjudicado"
]].head(20)

,id_contrato,referencia_del_contrato,estado_contrato,valor_del_contrato,proveedor_adjudicado
5,CO1.PCCNTR.9279238,1150-2026,Aprobado,NaN,NaN
6,CO1.PCCNTR.3241665,303-2022,Cerrado,NaN,NaN
9,CO1.PCCNTR.9272734,1141-2026,En ejecución,NaN,NaN
14,CO1.PCCNTR.3012807,1773-2021,Modificado,NaN,NaN
27,CO1.PCCNTR.8747985,2213-2025.,En ejecución,NaN,NaN
28,CO1.PCCNTR.5273883,2920-2023,terminado,NaN,NaN
31,CO1.PCCNTR.2192385,509-2021,Cerrado,NaN,NaN
37,CO1.PCCNTR.9272917,1144-2026,Borrador,NaN,NaN
38,CO1.PCCNTR.6561364,2639-2024,En ejecución,NaN,NaN
39,CO1.PCCNTR.9261166,1058-2026,Modificado,NaN,NaN


In [143]:
# Comparar los registros con faltantes en el bloque general frente al subconjunto sin información financiera

columnas_4906 = [
    "nombre_ordenador_del_gasto",
    "numero_de_cuenta",
    "tipo_de_cuenta",
    "el_contrato_puede_ser_prorrogado",
    "objeto_del_contrato",
    "nombre_del_banco"
]

sin_bloque_general = df[columnas_4906].isna().all(axis=1)

columnas_financieras = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia"
]

sin_financiera = df[columnas_financieras].isna().all(axis=1)

print("Registros con bloque general faltante:", sin_bloque_general.sum())
print("Registros con bloque financiero faltante:", sin_financiera.sum())

print(
    "Registros presentes en ambos subconjuntos:",
    (sin_bloque_general & sin_financiera).sum()
)

print(
    "Registros con bloque general faltante pero información financiera:",
    (sin_bloque_general & ~sin_financiera).sum()
)

print(
    "Registros con información general pero bloque financiero faltante:",
    (~sin_bloque_general & sin_financiera).sum()
)

Registros con bloque general faltante: 4906
Registros con bloque financiero faltante: 4887
Registros presentes en ambos subconjuntos: 4887
Registros con bloque general faltante pero información financiera: 19
Registros con información general pero bloque financiero faltante: 0


In [144]:
# Estados de los registros con bloque general completamente faltante

print(
    df.loc[sin_bloque_general, "estado_contrato"]
      .value_counts(dropna=False)
)

estado_contrato
Cerrado              1653
Modificado           1306
En ejecución          794
terminado             470
Borrador              308
Suspendido            139
Aprobado               99
Cancelado              51
enviado Proveedor      51
cedido                 31
En aprobación           4
Name: count, dtype: int64


In [145]:
# Características básicas del subconjunto

print("Registros:", sin_bloque_general.sum())

print(
    "\nFecha de firma disponibles:",
    df.loc[sin_bloque_general, "fecha_de_firma"].notna().sum()
)

print(
    "Fecha de inicio disponibles:",
    df.loc[sin_bloque_general, "fecha_de_inicio_del_contrato"].notna().sum()
)

print(
    "Fecha de fin disponibles:",
    df.loc[sin_bloque_general, "fecha_de_fin_del_contrato"].notna().sum()
)

Registros: 4906

Fecha de firma disponibles: 18
Fecha de inicio disponibles: 18
Fecha de fin disponibles: 19


In [146]:
# Analizar el tipo de contrato en el subconjunto con faltantes estructurales

print(
    df.loc[sin_bloque_general, "tipo_de_contrato"]
      .value_counts(dropna=False)
)

tipo_de_contrato
NaN                        4887
Prestación de servicios      17
Interventoría                 2
Name: count, dtype: int64


In [147]:
# Comparar tipo de contrato entre el subconjunto con faltantes y el conjunto completo

print("Subconjunto con faltantes estructurales:")
print(
    df.loc[sin_bloque_general, "tipo_de_contrato"]
      .value_counts(normalize=True, dropna=False)
      .head(10)
)

print("\nConjunto completo:")
print(
    df["tipo_de_contrato"]
      .value_counts(normalize=True, dropna=False)
      .head(10)
)

Subconjunto con faltantes estructurales:
tipo_de_contrato
NaN                        0.996127
Prestación de servicios    0.003465
Interventoría              0.000408
Name: proportion, dtype: float64

Conjunto completo:
tipo_de_contrato
Prestación de servicios    0.470924
NaN                        0.190861
Obra                       0.133138
Otro                       0.090842
Interventoría              0.059754
Suministros                0.023160
Consultoría                0.022378
Compraventa                0.003163
Comodato                   0.002890
Seguros                    0.001250
Name: proportion, dtype: float64


In [148]:
# Verificar si los subconjuntos coinciden exactamente

ids_sin_bloque_general = set(
    df.loc[sin_bloque_general, "id_contrato"]
)

ids_sin_financiera = set(
    df.loc[sin_financiera, "id_contrato"]
)

print(
    "IDs presentes en ambos subconjuntos:",
    len(ids_sin_bloque_general & ids_sin_financiera)
)

print(
    "IDs solo en bloque general:",
    len(ids_sin_bloque_general - ids_sin_financiera)
)

print(
    "IDs solo en bloque financiero:",
    len(ids_sin_financiera - ids_sin_bloque_general)
)

IDs presentes en ambos subconjuntos: 4887
IDs solo en bloque general: 19
IDs solo en bloque financiero: 0


### 2.15 Análisis y caracterización de faltantes estructurales

Se identificó un subconjunto de 4.906 registros con ausencia simultánea de información en múltiples variables contractuales.

De estos registros, 4.887 presentan además ausencia completa de las variables financieras analizadas. La comparación mediante `id_contrato` confirmó que los 4.887 registros pertenecen al mismo subconjunto en ambos análisis, mientras que existen 19 registros adicionales que presentan faltantes estructurales pero conservan información financiera.

El análisis de `tipo_de_contrato` mostró que el 99,61 % de los registros con faltantes estructurales presentan esta variable como faltante, frente al 19,09 % observado en el conjunto completo.

Los registros con faltantes estructurales se encuentran distribuidos entre diferentes estados contractuales, por lo que no se consideran exclusivos de contratos en estado Borrador o Cancelado.

**Conclusión:** los faltantes presentan un patrón estructural y altamente concentrado en un subconjunto específico de registros. No se realizará imputación automática de estos valores, debido a que no existe evidencia suficiente para determinar los valores correctos. Los registros serán conservados y su cobertura será considerada en los análisis posteriores.

### 2.16 Revisión de consistencia en variables categóricas

Una vez analizados los valores faltantes y los valores especiales, se procede a revisar la consistencia de las variables categóricas.

En esta etapa se busca identificar posibles inconsistencias en la forma en que se registran las categorías, como diferencias en mayúsculas y minúsculas, espacios adicionales o categorías que representen conceptualmente el mismo estado.

Se inicia la revisión con la variable `estado_contrato`, debido a su importancia para caracterizar la situación de los contratos y para realizar análisis posteriores.

El objetivo de esta revisión es determinar si las categorías observadas corresponden a estados contractuales diferentes o si existen variantes de escritura que deban ser estandarizadas.

In [149]:
# Revisar categorías únicas de estado_contrato

print(df["estado_contrato"].unique())

<ArrowStringArray>
[          'Cerrado',      'En ejecución',         'Cancelado',
          'Aprobado',        'Modificado',         'terminado',
          'Borrador',            'cedido',     'En aprobación',
        'Suspendido', 'enviado Proveedor']
Length: 11, dtype: str


In [150]:
# Número de categorías

print("Número de categorías:", df["estado_contrato"].nunique())

Número de categorías: 11


In [151]:
# Frecuencia de cada categoría

print(
    df["estado_contrato"]
      .value_counts(dropna=False)
)

estado_contrato
Cerrado              9263
Modificado           5434
terminado            3416
En ejecución         2512
Borrador             2438
Aprobado              872
Cancelado             629
Suspendido            361
enviado Proveedor     301
En aprobación         287
cedido                 92
Name: count, dtype: int64


In [152]:
# Revisar espacios al inicio o final en estado_contrato

estados_con_espacios = df[
    df["estado_contrato"].astype("string").str.strip()
    != df["estado_contrato"]
]["estado_contrato"]

print("Registros con espacios adicionales:", len(estados_con_espacios))

print(estados_con_espacios.unique())

Registros con espacios adicionales: 0
<ArrowStringArray>
[]
Length: 0, dtype: str


### 2.17 Revisión de consistencia en la modalidad de contratación

Se revisará la variable `modalidad_de_contratacion` para identificar posibles inconsistencias en sus categorías, tales como diferencias en mayúsculas y minúsculas, espacios adicionales o categorías que puedan representar la misma modalidad de contratación.

El objetivo es determinar si se requiere algún proceso de estandarización o si los valores pueden conservarse en su forma original.

In [153]:
# Revisar categorías únicas

print(df["modalidad_de_contratacion"].unique())

<ArrowStringArray>
[                                       'Contratación directa',
                                                           nan,
                             'Licitación pública Obra Publica',
                        'Selección Abreviada de Menor Cuantía',
                                              'Mínima cuantía',
                                          'Licitación pública',
                                 'Concurso de méritos abierto',
 'Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes',
                         'Selección abreviada subasta inversa',
                          'Contratación Directa (con ofertas)',
                               'Contratación régimen especial',
                 'Contratación régimen especial (con ofertas)']
Length: 12, dtype: str


In [154]:
# Número de categorías

print(
    "Número de categorías:",
    df["modalidad_de_contratacion"].nunique()
)

Número de categorías: 11


In [155]:
# Frecuencia de cada categoría

print(
    df["modalidad_de_contratacion"]
      .value_counts(dropna=False)
)

modalidad_de_contratacion
Contratación directa                                           14377
NaN                                                             4887
Mínima cuantía                                                  2909
Concurso de méritos abierto                                     1297
Licitación pública Obra Publica                                  787
Selección Abreviada de Menor Cuantía                             724
Licitación pública                                               429
Selección abreviada subasta inversa                              101
Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes       52
Contratación Directa (con ofertas)                                33
Contratación régimen especial                                      8
Contratación régimen especial (con ofertas)                        1
Name: count, dtype: int64


In [156]:
# Revisar espacios adicionales en modalidad_de_contratacion

modalidades_con_espacios = df[
    df["modalidad_de_contratacion"].astype("string").str.strip()
    != df["modalidad_de_contratacion"]
]["modalidad_de_contratacion"]

print("Registros con espacios adicionales:", len(modalidades_con_espacios))

print(modalidades_con_espacios.unique())

Registros con espacios adicionales: 0
<ArrowStringArray>
[]
Length: 0, dtype: str


In [157]:
# Comparar categorías normalizando temporalmente mayúsculas/minúsculas y espacios

modalidades_normalizadas = (
    df["modalidad_de_contratacion"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(
    modalidades_normalizadas
    .value_counts(dropna=False)
)

modalidad_de_contratacion
contratación directa                                           14377
<NA>                                                            4887
mínima cuantía                                                  2909
concurso de méritos abierto                                     1297
licitación pública obra publica                                  787
selección abreviada de menor cuantía                             724
licitación pública                                               429
selección abreviada subasta inversa                              101
seleccion abreviada menor cuantia sin manifestacion interes       52
contratación directa (con ofertas)                                33
contratación régimen especial                                      8
contratación régimen especial (con ofertas)                        1
Name: count, dtype: int64[pyarrow]


### 2.18 Revisión de consistencia en tipo de contrato

Se revisará la variable `tipo_de_contrato` para identificar posibles inconsistencias en sus categorías, tales como diferencias de mayúsculas y minúsculas, espacios adicionales o categorías que puedan representar el mismo tipo de contrato.

Esta revisión se realiza sobre el conjunto completo de registros. Los valores faltantes ya fueron caracterizados previamente como parte del análisis de faltantes estructurales.

El objetivo es determinar si las categorías existentes requieren algún proceso de estandarización, evitando realizar transformaciones que puedan alterar el significado original de los datos.

In [158]:
# Revisar categorías únicas de tipo_de_contrato

print(df["tipo_de_contrato"].unique())

<ArrowStringArray>
[   'Prestación de servicios',                          nan,
                       'Obra',                       'Otro',
              'Interventoría',                'Suministros',
   'Arrendamiento de muebles',                'Consultoría',
 'Arrendamiento de inmuebles',                'Compraventa',
                    'Seguros',                   'Comodato',
      'Servicios financieros',   'Acuerdo Marco de Precios',
                  'Concesión',        'Decreto 092 de 2017']
Length: 16, dtype: str


In [159]:
# Número de categorías

print(
    "Número de categorías:",
    df["tipo_de_contrato"].nunique()
)

Número de categorías: 15


In [160]:
# Frecuencia de cada categoría

print(
    df["tipo_de_contrato"]
      .value_counts(dropna=False)
)

tipo_de_contrato
Prestación de servicios       12058
NaN                            4887
Obra                           3409
Otro                           2326
Interventoría                  1530
Suministros                     593
Consultoría                     573
Compraventa                      81
Comodato                         74
Seguros                          32
Arrendamiento de inmuebles       31
Acuerdo Marco de Precios          5
Concesión                         2
Decreto 092 de 2017               2
Arrendamiento de muebles          1
Servicios financieros             1
Name: count, dtype: int64


In [161]:
# Revisar espacios adicionales en tipo_de_contrato

tipos_con_espacios = df[
    df["tipo_de_contrato"].astype("string").str.strip()
    != df["tipo_de_contrato"]
]["tipo_de_contrato"]

print("Registros con espacios adicionales:", len(tipos_con_espacios))

print(tipos_con_espacios.unique())

Registros con espacios adicionales: 0
<ArrowStringArray>
[]
Length: 0, dtype: str


In [162]:
# Comparar categorías ignorando temporalmente mayúsculas, minúsculas y espacios

tipos_normalizados = (
    df["tipo_de_contrato"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(
    tipos_normalizados
    .value_counts(dropna=False)
)

tipo_de_contrato
prestación de servicios       12058
<NA>                           4887
obra                           3409
otro                           2326
interventoría                  1530
suministros                     593
consultoría                     573
compraventa                      81
comodato                         74
seguros                          32
arrendamiento de inmuebles       31
acuerdo marco de precios          5
concesión                         2
decreto 092 de 2017               2
arrendamiento de muebles          1
servicios financieros             1
Name: count, dtype: int64[pyarrow]


### 2.19 Revisión de consistencia en ciudad

Se revisará la variable `ciudad` para identificar posibles inconsistencias en la forma en que se registran las ciudades de ejecución o localización de los contratos.

La revisión buscará identificar diferencias de formato, como espacios adicionales, variaciones en mayúsculas y minúsculas, y posibles categorías duplicadas que representen una misma ciudad.

También se evaluará la presencia de valores faltantes o poco informativos.

El objetivo es determinar si la variable requiere algún proceso de estandarización antes de utilizarla en análisis posteriores, conservando los valores originales cuando no exista evidencia suficiente para realizar una transformación.

In [163]:
# Número de valores únicos de ciudad

print("Número de categorías:", df["ciudad"].nunique(dropna=False))

Número de categorías: 1


In [164]:
# Frecuencia de las ciudades

print(
    df["ciudad"]
      .value_counts(dropna=False)
      .head(30)
)

ciudad
Bogotá    25605
Name: count, dtype: int64


In [165]:
# Verificar proporción de la única categoría

print(
    df["ciudad"]
      .value_counts(normalize=True, dropna=False)
)

ciudad
Bogotá    1.0
Name: proportion, dtype: float64


### 2.19.1 Decisión sobre la variable ciudad

La variable `ciudad` presenta una única categoría, correspondiente a `Bogotá`, en los 25.605 registros del conjunto de datos. La proporción observada es del 100 %, por lo que la variable no presenta variabilidad.

Debido a que una variable constante no aporta capacidad discriminatoria para los análisis posteriores, se decide eliminar `ciudad` del conjunto de datos de trabajo.

Esta decisión no implica que los datos de ciudad sean incorrectos; únicamente significa que la variable no aporta información diferencial dentro del conjunto analizado.

In [166]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["ciudad"])

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Número de filas: 25605
Número de columnas: 88


### 2.20 Revisión de consistencia en proveedor adjudicado

Se revisará la variable `proveedor_adjudicado`, que identifica al proveedor adjudicado en cada contrato.

Esta variable presenta un número elevado de categorías y una proporción importante de valores faltantes, por lo que se analizará la distribución de sus valores y la presencia de registros especiales como `Sin Descripcion`.

La revisión buscará identificar posibles inconsistencias de formato, valores especiales y posibles duplicidades derivadas de diferencias en la escritura de los nombres de los proveedores.

No se realizarán modificaciones sobre la variable hasta contar con evidencia suficiente que permita determinar la naturaleza de las inconsistencias encontradas.

In [167]:
print("Número de categorías:", df["proveedor_adjudicado"].nunique(dropna=False))

Número de categorías: 9159


In [168]:
print(
    df["proveedor_adjudicado"]
      .value_counts(dropna=False)
      .head(20)
)

proveedor_adjudicado
NaN                                                            5678
RAFAEL HERNAN RODRIGUEZ PRIETO                                   59
RIAC SAS                                                         47
UCING SAS                                                        40
INGDECOL S.A.S.                                                  34
AMV CONSULTORES SAS                                              33
Proyectos de Ingeniería Cumbre S.A.S.                            29
CYCOV SAS                                                        28
ALBERTO RAFAEL TOLEDO VERGARA                                    28
R&R PROYECTOS DE INGENIERIA SAS                                  27
TULCAN L. J. EIVAR                                               25
MARAMBAIA NEGOCIOS A TU ALCANCE S.A.S                            24
CARLOS ALBERTO SANCHEZ LACOUTURE                                 22
ALVARO FERNANDO LARA ZAMBRANO                                    22
COOPERATIVA DE TRABAJO ASOC

In [169]:
# Revisar espacios adicionales en proveedor_adjudicado

proveedores_con_espacios = df[
    df["proveedor_adjudicado"].astype("string").str.strip()
    != df["proveedor_adjudicado"]
]["proveedor_adjudicado"]

print("Registros con espacios adicionales:", len(proveedores_con_espacios))

print(proveedores_con_espacios.unique())

Registros con espacios adicionales: 0
<ArrowStringArray>
[]
Length: 0, dtype: str


In [170]:
# Normalización temporal para identificar posibles duplicados por formato

proveedores_normalizados = (
    df["proveedor_adjudicado"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(
    "Categorías originales:",
    df["proveedor_adjudicado"].nunique()
)

print(
    "Categorías después de normalizar:",
    proveedores_normalizados.nunique()
)

Categorías originales: 9158
Categorías después de normalizar: 9147


In [171]:
# Identificar proveedores que tienen más de una representación después de normalizar mayúsculas/minúsculas y espacios

comparacion_proveedores = pd.DataFrame({
    "original": df["proveedor_adjudicado"],
    "normalizado": proveedores_normalizados
})

duplicados_formato = (
    comparacion_proveedores
    .dropna()
    .groupby("normalizado")["original"]
    .nunique()
)

duplicados_formato = duplicados_formato[
    duplicados_formato > 1
]

print("Proveedores con múltiples representaciones:", len(duplicados_formato))
print(duplicados_formato)

Proveedores con múltiples representaciones: 11
normalizado
carlos humberto rincon rios                        2
consorcio unidos de zipa                           2
instituto nacional de vías                         2
juan david                                         2
junta de accion comunal                            2
junta de accion comunal de la vereda santa rita    2
junta de accion comunal vereda bellavista          2
junta de accion comunal vereda el diamante         2
junta de accion comunal vereda san luis            2
maria alejandra                                    2
santiago ramirez                                   2
Name: original, dtype: int64


In [172]:
# Mostrar las diferentes representaciones originales para cada proveedor normalizado

for proveedor in duplicados_formato.index:
    print("\n---", proveedor, "---")
    
    print(
        comparacion_proveedores[
            comparacion_proveedores["normalizado"] == proveedor
        ]["original"]
        .unique()
    )


--- carlos humberto rincon rios ---
<ArrowStringArray>
['Carlos Humberto Rincon Rios', 'CARLOS HUMBERTO RINCON RIOS']
Length: 2, dtype: str

--- consorcio unidos de zipa ---
<ArrowStringArray>
['consorcio Unidos de zipa', 'Consorcio unidos de zipa']
Length: 2, dtype: str

--- instituto nacional de vías ---
<ArrowStringArray>
['INSTITUTO NACIONAL DE VÍAS', 'Instituto Nacional de Vías']
Length: 2, dtype: str

--- juan david ---
<ArrowStringArray>
['JUAN DAVID', 'Juan David']
Length: 2, dtype: str

--- junta de accion comunal ---
<ArrowStringArray>
['JUNTA DE ACCION COMUNAL', 'junta de accion comunal']
Length: 2, dtype: str

--- junta de accion comunal de la vereda santa rita ---
<ArrowStringArray>
['Junta de accion comunal de la vereda Santa Rita', 'JUNTA DE ACCION COMUNAL DE LA VEREDA SANTA RITA']
Length: 2, dtype: str

--- junta de accion comunal vereda bellavista ---
<ArrowStringArray>
['JUNTA DE ACCION COMUNAL VEREDA BELLAVISTA', 'junta de accion comunal vereda bellavista']
Length: 

In [173]:
# Estandarizar formato de proveedor adjudicado

df["proveedor_adjudicado"] = (
    df["proveedor_adjudicado"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [174]:
# Verificar número de categorías después de la estandarización

print(
    "Número de categorías después de estandarizar:",
    df["proveedor_adjudicado"].nunique()
)

Número de categorías después de estandarizar: 9147


In [175]:
# Validar que la estandarización no alteró la estructura del conjunto

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

print(
    "Valores faltantes en proveedor_adjudicado:",
    df["proveedor_adjudicado"].isna().sum()
)

print(
    "Categorías finales:",
    df["proveedor_adjudicado"].nunique()
)

Filas: 25605
Columnas: 88
Valores faltantes en proveedor_adjudicado: 5678
Categorías finales: 9147


### 2.21 Revisión de consistencia en nombre de entidad

Se revisará la variable `nombre_entidad`, correspondiente al nombre de la entidad que realiza la contratación.

La revisión buscará identificar posibles inconsistencias de formato, como espacios adicionales, diferencias en mayúsculas y minúsculas, y posibles categorías duplicadas derivadas de distintas representaciones textuales de una misma entidad.

También se evaluará la presencia de valores faltantes o valores especiales.

El objetivo es determinar si la variable requiere estandarización para evitar que diferencias de formato generen categorías artificialmente distintas en los análisis posteriores.

In [176]:
# Número de categorías de nombre_entidad

print(
    "Número de categorías:",
    df["nombre_entidad"].nunique(dropna=False)
)

Número de categorías: 1


In [177]:
# Frecuencia de las entidades

print(
    df["nombre_entidad"]
      .value_counts(dropna=False)
      .head(20)
)

nombre_entidad
INVIAS    25605
Name: count, dtype: int64


In [178]:
# Verificar proporción de la única categoría

print(
    df["nombre_entidad"]
      .value_counts(normalize=True, dropna=False)
)

nombre_entidad
INVIAS    1.0
Name: proportion, dtype: float64


### 2.21.1 Decisión sobre la variable nombre_entidad

La variable `nombre_entidad` presenta una única categoría, correspondiente a `INVIAS`, en los 25.605 registros del conjunto de datos. La proporción observada es del 100 %, por lo que la variable no presenta variabilidad.

Debido a que una variable constante no aporta información diferenciadora para los análisis posteriores, se decide eliminar `nombre_entidad` del conjunto de datos de trabajo.

Esta decisión no implica que el valor registrado sea incorrecto. La variable se elimina únicamente porque todos los registros pertenecen a la misma entidad y, por tanto, no permite establecer diferencias entre contratos.

In [179]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["nombre_entidad"])

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Número de filas: 25605
Número de columnas: 87


### 2.22 Revisión de consistencia en nit_entidad

Se revisará la variable `nit_entidad`, correspondiente al identificador tributario de la entidad contratante.

La revisión busca determinar si existe variabilidad en los valores registrados, presencia de valores faltantes o posibles inconsistencias en su representación.

Dado que el conjunto de datos fue delimitado a contratos de INVIAS, se espera que el NIT corresponda a una única entidad. Por esta razón, una eventual variabilidad en esta variable requeriría una revisión adicional.

In [180]:
# Exploración de nit_entidad

print("Número de categorías:", df["nit_entidad"].nunique(dropna=False))

print("\nValores de nit_entidad:")
print(df["nit_entidad"].value_counts(dropna=False))

Número de categorías: 1

Valores de nit_entidad:
nit_entidad
800215807    25605
Name: count, dtype: int64


### 2.22.1 Decisión sobre la variable nit_entidad

La variable `nit_entidad` presenta una única categoría, correspondiente al valor `800215807`, presente en los 25.605 registros analizados. No se identificaron valores faltantes ni valores alternativos.

El resultado es consistente con la delimitación del proyecto, cuyo universo de análisis corresponde exclusivamente a INVIAS.

Aunque el NIT constituye un identificador válido de la entidad, no presenta variabilidad dentro del conjunto de estudio y, por tanto, no aporta capacidad diferenciadora para los análisis posteriores. En consecuencia, se elimina del conjunto de datos de trabajo, manteniendo documentado que el universo analizado corresponde a un único NIT.

In [181]:
df = df.drop(columns=["nit_entidad"])

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 25605
Columnas: 86


### 2.23 Revisión de la variable localizacion

La variable `localizacion` registra la ubicación asociada a la entidad contratante. En el conjunto de datos analizado, correspondiente exclusivamente a INVIAS, presenta un único valor: `Colombia, Bogotá, Bogotá`, para los 25.605 registros.

Esta característica es consistente con que INVIAS es la entidad contratante del universo seleccionado. Sin embargo, la variable no permite identificar el lugar donde se ejecutan los contratos y, adicionalmente, no presenta variabilidad dentro del conjunto de datos.

Dado que la pregunta de investigación #3 se enfoca en la ejecución financiera y las modificaciones contractuales, `localizacion` no aporta información diferenciadora para los análisis requeridos.

Por estas razones, se decide eliminar esta variable del conjunto de datos de trabajo.

In [182]:
print("Número de categorías:", df["localizacion"].nunique(dropna=False))

print("\nValores más frecuentes:")
print(
    df["localizacion"]
      .value_counts(dropna=False)
      .head(20)
)

Número de categorías: 1

Valores más frecuentes:
localizacion
Colombia, Bogotá,  Bogotá    25605
Name: count, dtype: int64


In [183]:
df = df.drop(columns=["localizacion"])

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 25605
Columnas: 85


### 2.24 Revisión de la variable orden

Se revisará la variable `orden` para identificar el número de categorías presentes, la distribución de sus valores y la existencia de posibles valores faltantes.

El objetivo es determinar si esta variable aporta información diferenciadora dentro del conjunto de contratos de INVIAS y si resulta pertinente para los análisis asociados a la ejecución financiera y las modificaciones contractuales.

In [184]:
print("Número de categorías:", df["orden"].nunique(dropna=False))

print("\nValores de orden:")
print(df["orden"].value_counts(dropna=False))

Número de categorías: 1

Valores de orden:
orden
Nacional    25605
Name: count, dtype: int64


### 2.24.1 Decisión sobre la variable orden

La variable `orden` presenta una única categoría, correspondiente a `Nacional`, presente en los 25.605 registros analizados. No se identificaron valores faltantes ni categorías alternativas.

Esta característica es consistente con la naturaleza de INVIAS como entidad del orden nacional. Sin embargo, la variable no presenta variabilidad dentro del conjunto de datos y no aporta información diferenciadora para responder la pregunta de investigación relacionada con la ejecución financiera y las modificaciones contractuales.

En consecuencia, se decide eliminar `orden` del conjunto de datos de trabajo.

In [185]:
df = df.drop(columns=["orden"])

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 25605
Columnas: 84


### 2.25 Revisión de la variable sector

Se revisará la variable `sector` para identificar las categorías presentes, su frecuencia y la existencia de valores faltantes.

El objetivo es determinar si la variable presenta variabilidad dentro del conjunto de contratos de INVIAS y evaluar su posible utilidad como variable de contexto para los análisis posteriores.

In [186]:
print("Número de categorías:", df["sector"].nunique(dropna=False))

print("\nDistribución de sector:")
print(df["sector"].value_counts(dropna=False))

Número de categorías: 1

Distribución de sector:
sector
Transporte    25605
Name: count, dtype: int64


### 2.25.1 Decisión sobre la variable sector

La variable `sector` presenta una única categoría, correspondiente a `Transporte`, presente en los 25.605 registros analizados. No se identificaron valores faltantes ni categorías alternativas.

La ausencia de variabilidad es consistente con la naturaleza de la entidad analizada y con la delimitación del conjunto de datos a contratos de INVIAS. Sin embargo, la variable no permite diferenciar los contratos y no aporta información adicional para el análisis de la ejecución financiera y las modificaciones contractuales.

En consecuencia, se decide eliminar `sector` del conjunto de datos de trabajo.

In [187]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["sector"])

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 25605
Columnas: 83


### 2.26 Revisión de la variable rama

Se revisará la variable `rama` para identificar las categorías presentes, su distribución y la existencia de valores faltantes.

El objetivo es determinar si presenta variabilidad dentro del conjunto de contratos de INVIAS y evaluar si aporta información diferenciadora para el análisis.

In [188]:
print("Número de categorías:", df["rama"].nunique(dropna=False))

print("\nDistribución de rama:")
print(df["rama"].value_counts(dropna=False))

Número de categorías: 1

Distribución de rama:
rama
Ejecutivo    25605
Name: count, dtype: int64


### 2.26.1 Decisión sobre la variable rama

La variable `rama` presenta una única categoría, correspondiente a `Ejecutivo`, presente en los 25.605 registros analizados. No se identificaron valores faltantes ni categorías alternativas.

La ausencia de variabilidad indica que esta variable no permite diferenciar los contratos dentro del universo de estudio. Adicionalmente, no aporta información diferenciadora para el análisis de la ejecución financiera y las modificaciones contractuales planteado en la pregunta de investigación.

En consecuencia, se decide eliminar `rama` del conjunto de datos de trabajo.

In [189]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["rama"])

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 25605
Columnas: 82


### 2.27 Revisión de la variable entidad_centralizada

Se revisará la variable `entidad_centralizada` para identificar las categorías presentes, su distribución y la existencia de valores faltantes.

El objetivo es determinar si presenta variabilidad dentro del conjunto de contratos de INVIAS y evaluar su utilidad como variable de contexto para los análisis posteriores.

In [190]:
print("Número de categorías:", df["entidad_centralizada"].nunique(dropna=False))

print("\nDistribución de entidad_centralizada:")
print(df["entidad_centralizada"].value_counts(dropna=False))

Número de categorías: 1

Distribución de entidad_centralizada:
entidad_centralizada
Centralizada    25605
Name: count, dtype: int64


### 2.27.1 Decisión sobre la variable entidad_centralizada

La variable `entidad_centralizada` presenta una única categoría correspondiente a `Centralizada`, presente en los 25.605 registros analizados.

Al no presentar variabilidad dentro del universo de estudio, la variable no permite diferenciar los contratos ni aporta información relevante para el análisis de la ejecución financiera y las modificaciones contractuales.

En consecuencia, se decide eliminar `entidad_centralizada` del conjunto de datos de trabajo.

In [191]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["entidad_centralizada"])

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 25605
Columnas: 81


### 2.28 Revisión de la variable proceso_de_compra

Se revisará la variable `proceso_de_compra` para identificar el número de categorías presentes, su distribución y la existencia de valores faltantes.

El objetivo es determinar la variabilidad de esta variable y evaluar su utilidad como identificador o variable de contexto dentro del análisis de los contratos de INVIAS.

In [192]:
print("Número de categorías:", df["proceso_de_compra"].nunique(dropna=False))

print("\nValores más frecuentes:")
print(
    df["proceso_de_compra"]
      .value_counts(dropna=False)
      .head(20)
)

Número de categorías: 22381

Valores más frecuentes:
proceso_de_compra
CO1.BDOS.927424     36
CO1.BDOS.4330907    30
CO1.BDOS.766571     26
CO1.BDOS.5248139    25
CO1.BDOS.284114     23
CO1.BDOS.926213     22
CO1.BDOS.282626     21
CO1.BDOS.749750     21
CO1.BDOS.927313     21
CO1.BDOS.2457753    17
CO1.BDOS.757656     17
CO1.BDOS.5243100    16
CO1.BDOS.285227     16
CO1.BDOS.1055556    16
CO1.BDOS.4373534    15
CO1.BDOS.942840     15
CO1.BDOS.758519     15
CO1.BDOS.283433     15
CO1.BDOS.282809     15
CO1.BDOS.283176     14
Name: count, dtype: int64


### 2.28.1 Relación entre proceso de compra e id_contrato

Dado que `proceso_de_compra` presenta un número elevado de categorías, se revisará su relación con `id_contrato` para determinar si un mismo proceso de compra puede estar asociado a uno o varios contratos.

Esta revisión permitirá establecer si la variable funciona principalmente como identificador del proceso de contratación y evaluar su utilidad para los análisis posteriores.

In [193]:
# Número de contratos asociados a cada proceso de compra

contratos_por_proceso = (
    df.groupby("proceso_de_compra")["id_contrato"]
      .nunique()
)

print("Procesos de compra:", contratos_por_proceso.shape[0])

print("\nProcesos asociados a más de un contrato:")
print(
    contratos_por_proceso[
        contratos_por_proceso > 1
    ].sort_values(ascending=False).head(20)
)

Procesos de compra: 22381

Procesos asociados a más de un contrato:
proceso_de_compra
CO1.BDOS.927424     36
CO1.BDOS.4330907    30
CO1.BDOS.766571     26
CO1.BDOS.5248139    25
CO1.BDOS.284114     23
CO1.BDOS.926213     22
CO1.BDOS.282626     21
CO1.BDOS.749750     21
CO1.BDOS.927313     21
CO1.BDOS.757656     17
CO1.BDOS.2457753    17
CO1.BDOS.5243100    16
CO1.BDOS.285227     16
CO1.BDOS.1055556    16
CO1.BDOS.4373534    15
CO1.BDOS.758519     15
CO1.BDOS.942840     15
CO1.BDOS.283433     15
CO1.BDOS.282809     15
CO1.BDOS.2761381    14
Name: id_contrato, dtype: int64


### 2.28.2 Decisión sobre la variable proceso_de_compra

La variable `proceso_de_compra` presenta 22.381 categorías diferentes dentro de los 25.605 registros analizados, evidenciando una alta variabilidad.

La revisión de su relación con `id_contrato` muestra que un mismo proceso de compra puede estar asociado a múltiples contratos. Por ejemplo, el proceso `CO1.BDOS.927424` se encuentra asociado a 36 contratos diferentes.

Aunque la variable no constituye una medida directa de la ejecución financiera ni de las modificaciones contractuales, puede aportar información de trazabilidad y permitir relacionar contratos que pertenecen a un mismo proceso de contratación.

Por esta razón, se decide conservar `proceso_de_compra` en el conjunto de datos de trabajo, sin utilizarla como variable analítica principal.

### 2.29 Clasificación de variables según su utilidad analítica

Las variables restantes del conjunto de datos se clasifican de acuerdo con su función dentro de la pregunta de investigación.

La clasificación permite priorizar el proceso de limpieza y análisis, concentrando la revisión detallada en las variables directamente relacionadas con la ejecución financiera y las modificaciones contractuales.

Se establecen las siguientes categorías:

- **Financiera:** variables utilizadas para medir la ejecución, pagos, saldos y recursos pendientes.
- **Temporal:** variables relacionadas con fechas, duración y modificaciones de plazo.
- **Contractual:** variables utilizadas para caracterizar el contrato y su estado.
- **Contexto:** variables que permiten segmentar o contextualizar los resultados.
- **Identificación:** variables necesarias para identificar y dar trazabilidad a los contratos.
- **Metadatos:** variables asociadas con el origen, actualización o características técnicas del registro.
- **Revisión:** variables cuya utilidad debe determinarse mediante una revisión puntual.
- **No relevante:** variables que no aportan información necesaria para responder la pregunta de investigación.

In [194]:
# Inventario actual de variables

for i, col in enumerate(df.columns, start=1):
    print(i, "-", col)

1 - id
2 - version
3 - created_at
4 - updated_at
5 - proceso_de_compra
6 - id_contrato
7 - referencia_del_contrato
8 - estado_contrato
9 - codigo_de_categoria_principal
10 - descripcion_del_proceso
11 - tipo_de_contrato
12 - modalidad_de_contratacion
13 - justificacion_modalidad_de
14 - fecha_de_firma
15 - fecha_de_inicio_del_contrato
16 - fecha_de_fin_del_contrato
17 - condiciones_de_entrega
18 - tipodocproveedor
19 - documento_proveedor
20 - proveedor_adjudicado
21 - es_grupo
22 - es_pyme
23 - habilita_pago_adelantado
24 - liquidacion
25 - obligacion_ambiental
26 - obligaciones_postconsumo
27 - reversion
28 - origen_de_los_recursos
29 - destino_gasto
30 - valor_del_contrato
31 - valor_de_pago_adelantado
32 - valor_facturado
33 - valor_pendiente_de_pago
34 - valor_pagado
35 - valor_amortizado
36 - valor_pendiente_de
37 - valor_pendiente_de_ejecucion
38 - saldo_cdp
39 - saldo_vigencia
40 - espostconflicto
41 - dias_adicionados
42 - puntos_del_acuerdo
43 - pilares_del_acuerdo
44 - urlpr

In [195]:
clasificacion_variables = {

    # -------------------------
    # FINANCIERAS
    # -------------------------
    "valor_del_contrato": "Financiera",
    "valor_de_pago_adelantado": "Financiera",
    "valor_facturado": "Financiera",
    "valor_pendiente_de_pago": "Financiera",
    "valor_pagado": "Financiera",
    "valor_amortizado": "Financiera",
    "valor_pendiente_de_ejecucion": "Financiera",
    "saldo_cdp": "Financiera",
    "saldo_vigencia": "Financiera",

    # -------------------------
    # TEMPORALES
    # -------------------------
    "fecha_de_firma": "Temporal",
    "fecha_de_inicio_del_contrato": "Temporal",
    "fecha_de_fin_del_contrato": "Temporal",
    "dias_adicionados": "Temporal",
    "duracion_del_contrato": "Temporal",
    "fecha_inicio_liquidacion": "Temporal",
    "fecha_fin_liquidacion": "Temporal",
    "fecha_de_notificacion_de_prorrogacion": "Temporal",

    # -------------------------
    # CONTRACTUALES
    # -------------------------
    "estado_contrato": "Contractual",
    "tipo_de_contrato": "Contractual",
    "modalidad_de_contratacion": "Contractual",
    "el_contrato_puede_ser_prorrogado": "Contractual",
    "liquidacion": "Contractual",
    "condiciones_de_entrega": "Contractual",
    "habilita_pago_adelantado": "Contractual",

    # -------------------------
    # CONTEXTO
    # -------------------------
    "proveedor_adjudicado": "Contexto",
    "es_grupo": "Contexto",
    "es_pyme": "Contexto",
    "origen_de_los_recursos": "Contexto",
    "destino_gasto": "Contexto",
    "objeto_del_contrato": "Contexto",
    "codigo_de_categoria_principal": "Contexto",
    "descripcion_del_proceso": "Contexto",
    "justificacion_modalidad_de": "Contexto",

    # -------------------------
    # IDENTIFICACIÓN
    # -------------------------
    "id": "Identificación",
    "id_contrato": "Identificación",
    "proceso_de_compra": "Identificación",
    "referencia_del_contrato": "Identificación",
    "codigo_entidad": "Identificación",
    "codigo_proveedor": "Identificación",
    "documento_proveedor": "Identificación",

    # -------------------------
    # METADATOS
    # -------------------------
    "version": "Metadatos",
    "created_at": "Metadatos",
    "updated_at": "Metadatos",
    "ultima_actualizacion": "Metadatos",
    "urlproceso": "Metadatos",

    # -------------------------
    # REVISIÓN
    # -------------------------
    "tipodocproveedor": "Revisión",
    "obligacion_ambiental": "Revisión",
    "obligaciones_postconsumo": "Revisión",
    "reversion": "Revisión",
    "espostconflicto": "Revisión",
    "puntos_del_acuerdo": "Revisión",
    "pilares_del_acuerdo": "Revisión",
    "presupuesto_general_de_la_nacion_pgn": "Revisión",
    "sistema_general_de_participaciones": "Revisión",
    "sistema_general_de_regal_as": "Revisión",
    "recursos_propios_alcald_as_gobernaciones_y_resguardos_ind_genas_": "Revisión",
    "recursos_de_credito": "Revisión",
    "recursos_propios": "Revisión",
    "inicio_antes_firma": "Revisión",
    "fin_antes_inicio": "Revisión",
    "valor_pendiente_de": "Revisión",

    # -------------------------
    # NO RELEVANTES
    # -------------------------
    "nombre_representante_legal": "No relevante",
    "nacionalidad_representante_legal": "No relevante",
    "domicilio_representante_legal": "No relevante",
    "tipo_de_identificaci_n_representante_legal": "No relevante",
    "identificaci_n_representante_legal": "No relevante",
    "genero_representante_legal": "No relevante",
    "nombre_del_banco": "No relevante",
    "tipo_de_cuenta": "No relevante",
    "numero_de_cuenta": "No relevante",
    "nombre_ordenador_del_gasto": "No relevante",
    "tipo_de_documento_ordenador_del_gasto": "No relevante",
    "numero_de_documento_ordenador_del_gasto": "No relevante",
    "nombre_supervisor": "No relevante",
    "tipo_de_documento_supervisor": "No relevante",
    "numero_de_documento_supervisor": "No relevante",
    "nombre_ordenador_de_pago": "No relevante",
    "tipo_de_documento_ordenador_de_pago": "No relevante",
    "numero_de_documento_ordenador_de_pago": "No relevante",
    "documentos_tipo": "No relevante",
    "descripcion_documentos_tipo": "No relevante"
}

In [196]:
# Verificar que todas las columnas actuales estén clasificadas

columnas_sin_clasificar = [
    col for col in df.columns
    if col not in clasificacion_variables
]

print("Columnas del dataset:", len(df.columns))
print("Columnas clasificadas:", len(clasificacion_variables))
print("Columnas sin clasificar:", len(columnas_sin_clasificar))

print("\nColumnas sin clasificar:")
print(columnas_sin_clasificar)

Columnas del dataset: 81
Columnas clasificadas: 81
Columnas sin clasificar: 0

Columnas sin clasificar:
[]


In [197]:
print(df["valor_pendiente_de"].head(20))

print("\nTipo de dato:")
print(df["valor_pendiente_de"].dtype)

print("\nValores faltantes:")
print(df["valor_pendiente_de"].isna().sum())

print("\nValores únicos:")
print(df["valor_pendiente_de"].nunique(dropna=False))

0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     NaN
6     NaN
7     0.0
8     0.0
9     NaN
10    0.0
11    0.0
12    0.0
13    0.0
14    NaN
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
Name: valor_pendiente_de, dtype: float64

Tipo de dato:
float64

Valores faltantes:
4887

Valores únicos:
23


In [198]:
print(
    df["valor_pendiente_de"]
      .value_counts(dropna=False)
      .sort_index()
)

valor_pendiente_de
0.000000e+00    20697
4.270700e+04        1
2.475000e+05        1
1.201259e+06        1
3.501195e+06        1
5.841403e+06        1
7.194264e+06        1
7.999115e+06        1
1.197428e+07        1
1.575075e+07        1
1.951293e+07        1
2.310882e+07        1
3.109818e+07        1
5.676343e+07        1
1.079603e+08        1
1.166702e+08        1
1.378433e+08        1
1.852858e+08        1
2.370890e+08        1
1.049216e+09        1
1.306397e+09        1
3.796304e+09        1
NaN              4887
Name: count, dtype: int64


In [199]:
print("valor_pendiente_de:")
print(df["valor_pendiente_de"].describe())

print("\nvalor_pendiente_de_ejecucion:")
print(df["valor_pendiente_de_ejecucion"].describe())

valor_pendiente_de:
count    2.071800e+04
mean     3.437109e+05
std      2.894542e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.796304e+09
Name: valor_pendiente_de, dtype: float64

valor_pendiente_de_ejecucion:
count    2.071800e+04
mean     1.837707e+09
std      1.953249e+10
min     -4.402933e+07
25%      3.334536e+06
50%      3.913187e+07
75%      1.998125e+08
max      6.571764e+11
Name: valor_pendiente_de_ejecucion, dtype: float64


### 2.30 Auditoría conjunta de variables financieras

Se realiza una auditoría conjunta de las variables financieras relevantes para la Pregunta 3, con el propósito de evaluar su completitud, distribución y consistencia antes de construir indicadores de ejecución financiera.

Las variables consideradas son:

- `valor_del_contrato`
- `valor_de_pago_adelantado`
- `valor_facturado`
- `valor_pendiente_de_pago`
- `valor_pagado`
- `valor_amortizado`
- `valor_pendiente_de`
- `valor_pendiente_de_ejecucion`
- `saldo_cdp`
- `saldo_vigencia`

La revisión considera valores faltantes, ceros, valores negativos y estadísticos descriptivos. Posteriormente se evaluará la consistencia entre las variables financieras antes de construir los indicadores de ejecución.

In [200]:
variables_financieras = [
    "valor_del_contrato",
    "valor_de_pago_adelantado",
    "valor_facturado",
    "valor_pendiente_de_pago",
    "valor_pagado",
    "valor_amortizado",
    "valor_pendiente_de",
    "valor_pendiente_de_ejecucion",
    "saldo_cdp",
    "saldo_vigencia"
]

resumen_financiero = pd.DataFrame({
    "tipo_dato": df[variables_financieras].dtypes,
    "faltantes": df[variables_financieras].isna().sum(),
    "ceros": (df[variables_financieras] == 0).sum(),
    "negativos": (df[variables_financieras] < 0).sum(),
    "unicos": df[variables_financieras].nunique(dropna=True)
})

resumen_financiero

,tipo_dato,faltantes,ceros,negativos,unicos
valor_del_contrato,float64,4887,1086,0,12360
valor_de_pago_adelantado,float64,4887,20697,0,22
valor_facturado,float64,4887,12396,0,6156
valor_pendiente_de_pago,float64,4887,2471,4,13529
valor_pagado,float64,4887,12591,0,6056
valor_amortizado,float64,4887,20717,0,2
valor_pendiente_de,float64,4887,20697,0,22
valor_pendiente_de_ejecucion,float64,4887,2486,4,13514
saldo_cdp,float64,4887,2503,0,4808
saldo_vigencia,float64,4887,19466,0,406


### 2.30.1 Revisión de valores negativos en variables financieras

La auditoría financiera identificó cuatro registros con valores negativos tanto en `valor_pendiente_de_pago` como en `valor_pendiente_de_ejecucion`.

Dado que estas variables representan recursos pendientes, los valores negativos requieren una revisión individual para determinar si corresponden a inconsistencias de los datos, ajustes financieros u otra situación particular del registro.

Se identificarán los contratos afectados y se compararán las principales variables financieras asociadas.

In [201]:
negativos_financieros = df[
    (df["valor_pendiente_de_pago"] < 0) |
    (df["valor_pendiente_de_ejecucion"] < 0)
]

print("Registros con valores negativos:", len(negativos_financieros))

negativos_financieros[
    [
        "id_contrato",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion",
        "saldo_cdp"
    ]
]

Registros con valores negativos: 4


,id_contrato,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,saldo_cdp
923,CO1.PCCNTR.5755707,13566666.66,57596000.0,57596000.0,-44029333.0,-44029333.0,6.030800e+09
3896,CO1.PCCNTR.828527,64544330.00,64750330.0,64750330.0,-206000.0,-206000.0,1.827747e+09
14383,CO1.PCCNTR.1300022,86948047.67,93386666.0,93386666.0,-6438619.0,-6438619.0,1.352610e+09
24534,CO1.PCCNTR.2174192,57141447.00,57170047.0,57170047.0,-28600.0,-28600.0,3.119647e+09


### Hallazgo

La auditoría identificó cuatro registros con valores negativos en `valor_pendiente_de_pago` y `valor_pendiente_de_ejecucion`.

La revisión individual muestra que en los cuatro casos el valor facturado y el valor pagado superan el `valor_del_contrato`. La magnitud del valor negativo en los campos pendientes corresponde a la diferencia entre el valor del contrato y el valor facturado/pagado.

Estos registros no serán modificados ni eliminados en esta etapa. Se conservarán para mantener la trazabilidad de la información original y serán considerados posteriormente en la validación de consistencia de los indicadores financieros.

### 2.30.2 Revisión de contratos con valor contractual igual a cero

La auditoría financiera identificó 1.086 registros cuyo `valor_del_contrato` es igual a cero.

Dado que esta variable constituye el denominador de varios indicadores de ejecución financiera, estos registros requieren una revisión específica antes de calcular porcentajes de ejecución o pago.

Se analizarán sus principales características contractuales y financieras para determinar si corresponden a registros válidos, contratos sin valor económico reportado o situaciones que deban excluirse de determinados indicadores.

In [202]:
contratos_valor_cero = df[df["valor_del_contrato"] == 0]

print("Contratos con valor = 0:", len(contratos_valor_cero))

contratos_valor_cero[
    [
        "id_contrato",
        "estado_contrato",
        "tipo_de_contrato",
        "modalidad_de_contratacion",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion",
        "saldo_cdp"
    ]
].head(20)

Contratos con valor = 0: 1086


,id_contrato,estado_contrato,tipo_de_contrato,modalidad_de_contratacion,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,saldo_cdp
4,CO1.PCCNTR.3516681,Cancelado,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,7.200000e+09
18,CO1.PCCNTR.6838297,Cancelado,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,1.803404e+10
49,CO1.PCCNTR.755108,Borrador,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,1.869300e+09
51,CO1.PCCNTR.8277719,Borrador,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,1.657125e+10
52,CO1.PCCNTR.7398046,Cancelado,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,1.460882e+09
59,CO1.PCCNTR.7766838,Cancelado,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,1.657575e+10
63,CO1.PCCNTR.8202459,En ejecución,Otro,Contratación directa,0.0,0.0,0.0,0.0,0.0,0.000000e+00
74,CO1.PCCNTR.6580260,En ejecución,Otro,Contratación directa,0.0,0.0,0.0,0.0,0.0,0.000000e+00
84,CO1.PCCNTR.8813998,Borrador,Prestación de servicios,Contratación directa,0.0,0.0,0.0,0.0,0.0,6.005814e+09
93,CO1.PCCNTR.5240629,terminado,Otro,Contratación directa,0.0,0.0,0.0,0.0,0.0,1.925000e+10


In [203]:
contratos_valor_cero = df[df["valor_del_contrato"] == 0]

print("Total contratos con valor = 0:", len(contratos_valor_cero))

print("\nEstados:")
print(
    contratos_valor_cero["estado_contrato"]
    .value_counts(dropna=False)
)

print("\nContratos con facturación > 0:")
print(
    (contratos_valor_cero["valor_facturado"] > 0).sum()
)

print("\nContratos con pagos > 0:")
print(
    (contratos_valor_cero["valor_pagado"] > 0).sum()
)

print("\nContratos con pendiente de ejecución > 0:")
print(
    (contratos_valor_cero["valor_pendiente_de_ejecucion"] > 0).sum()
)

print("\nContratos con pendiente de pago > 0:")
print(
    (contratos_valor_cero["valor_pendiente_de_pago"] > 0).sum()
)

Total contratos con valor = 0: 1086

Estados:
estado_contrato
Borrador             545
Cancelado            270
En ejecución          72
Modificado            54
terminado             49
Aprobado              45
enviado Proveedor     30
Cerrado               13
En aprobación          8
Name: count, dtype: int64

Contratos con facturación > 0:
0

Contratos con pagos > 0:
0

Contratos con pendiente de ejecución > 0:
0

Contratos con pendiente de pago > 0:
0


### Hallazgo y decisión metodológica

Se identificaron 1.086 registros con `valor_del_contrato` igual a cero.

La revisión mostró que ninguno de estos registros presenta valores positivos en `valor_facturado`, `valor_pagado`, `valor_pendiente_de_ejecucion` o `valor_pendiente_de_pago`.

Por lo tanto, no se evidencia en esta revisión una inconsistencia financiera derivada de estos registros. Los contratos se conservarán en el conjunto de datos para mantener la integridad y trazabilidad de la información.

Sin embargo, debido a que `valor_del_contrato` constituye el denominador de los indicadores porcentuales de ejecución y pago, estos registros serán excluidos únicamente de dichos cálculos cuando el denominador sea igual a cero. No serán eliminados del conjunto de datos general.

### 2.31 Consistencia entre variables financieras

Una vez revisados los valores faltantes, ceros y valores negativos, se evalúa la consistencia interna de las principales variables financieras del conjunto de datos.

Se analizan las relaciones entre el valor contractual, los valores facturados y pagados y los recursos pendientes de ejecución y de pago.

El objetivo es identificar posibles inconsistencias que puedan afectar la construcción posterior de los indicadores de ejecución financiera.

In [204]:
# Registros con valor contractual positivo
df_fin = df[df["valor_del_contrato"] > 0].copy()

print("Contratos con valor contractual > 0:", len(df_fin))

# 1. Facturado mayor que valor del contrato
facturado_mayor_contrato = (
    df_fin["valor_facturado"] > df_fin["valor_del_contrato"]
).sum()

# 2. Pagado mayor que facturado
pagado_mayor_facturado = (
    df_fin["valor_pagado"] > df_fin["valor_facturado"]
).sum()

# 3. Pagado mayor que valor del contrato
pagado_mayor_contrato = (
    df_fin["valor_pagado"] > df_fin["valor_del_contrato"]
).sum()

# 4. Pendiente de pago negativo
pendiente_pago_negativo = (
    df_fin["valor_pendiente_de_pago"] < 0
).sum()

# 5. Pendiente de ejecución negativo
pendiente_ejecucion_negativo = (
    df_fin["valor_pendiente_de_ejecucion"] < 0
).sum()

print("\n--- Consistencia financiera ---")
print("Facturado > valor del contrato:", facturado_mayor_contrato)
print("Pagado > facturado:", pagado_mayor_facturado)
print("Pagado > valor del contrato:", pagado_mayor_contrato)
print("Pendiente de pago < 0:", pendiente_pago_negativo)
print("Pendiente de ejecución < 0:", pendiente_ejecucion_negativo)

Contratos con valor contractual > 0: 19632

--- Consistencia financiera ---
Facturado > valor del contrato: 5
Pagado > facturado: 0
Pagado > valor del contrato: 5
Pendiente de pago < 0: 4
Pendiente de ejecución < 0: 4


In [205]:
casos_facturado_mayor = df_fin[
    df_fin["valor_facturado"] > df_fin["valor_del_contrato"]
]

casos_facturado_mayor[
    [
        "id_contrato",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion",
        "saldo_cdp"
    ]
]

,id_contrato,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,saldo_cdp
923,CO1.PCCNTR.5755707,13566666.66,57596000.0,57596000.0,-44029333.0,-44029333.0,6.030800e+09
3896,CO1.PCCNTR.828527,64544330.00,64750330.0,64750330.0,-206000.0,-206000.0,1.827747e+09
14383,CO1.PCCNTR.1300022,86948047.67,93386666.0,93386666.0,-6438619.0,-6438619.0,1.352610e+09
21957,CO1.PCCNTR.760010,41999999.66,42000000.0,42000000.0,0.0,0.0,6.319172e+08
24534,CO1.PCCNTR.2174192,57141447.00,57170047.0,57170047.0,-28600.0,-28600.0,3.119647e+09


### 2.32 Construcción de indicadores de ejecución financiera

A partir de las variables financieras previamente auditadas se construyen dos indicadores principales para evaluar el comportamiento financiero de los contratos:

- Porcentaje ejecutado: relación entre el valor facturado y el valor del contrato.
- Porcentaje pagado: relación entre el valor pagado y el valor del contrato.

Los indicadores se calcularán únicamente para los registros con `valor_del_contrato` positivo, debido a que los contratos con valor contractual igual a cero no permiten obtener un porcentaje de ejecución o pago definido.

Los valores originales de las variables financieras no serán modificados. Los casos en los que el valor facturado o pagado supera el valor contractual serán conservados para su posterior análisis y no serán corregidos de manera automática.

In [206]:
df["porcentaje_ejecutado"] = np.where(
    df["valor_del_contrato"] > 0,
    (df["valor_facturado"] / df["valor_del_contrato"]) * 100,
    np.nan
)

df["porcentaje_pagado"] = np.where(
    df["valor_del_contrato"] > 0,
    (df["valor_pagado"] / df["valor_del_contrato"]) * 100,
    np.nan
)

print("Porcentaje ejecutado creado:", "porcentaje_ejecutado" in df.columns)
print("Porcentaje pagado creado:", "porcentaje_pagado" in df.columns)

print("\nValores faltantes:")
print(df[["porcentaje_ejecutado", "porcentaje_pagado"]].isna().sum())

Porcentaje ejecutado creado: True
Porcentaje pagado creado: True

Valores faltantes:
porcentaje_ejecutado    5973
porcentaje_pagado       5973
dtype: int64


### 2.32.1 Validación de los indicadores de ejecución

Los indicadores de porcentaje ejecutado y porcentaje pagado presentan 5.973 valores faltantes cada uno.

Esta cantidad corresponde exactamente a la suma de los 4.887 registros con faltantes estructurales en las variables financieras y los 1.086 contratos cuyo valor contractual es igual a cero.

Por lo tanto, los valores faltantes de los indicadores están explicados por las condiciones previamente identificadas y no corresponden a nuevos faltantes generados durante el cálculo.

In [207]:
indicadores = [
    "porcentaje_ejecutado",
    "porcentaje_pagado"
]

print(df[indicadores].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))

print("\n--- Porcentaje ejecutado ---")
print("Igual a 0%:", (df["porcentaje_ejecutado"] == 0).sum())
print("Entre 0% y 100%:", (
    (df["porcentaje_ejecutado"] >= 0) &
    (df["porcentaje_ejecutado"] <= 100)
).sum())
print("Mayor a 100%:", (df["porcentaje_ejecutado"] > 100).sum())

print("\n--- Porcentaje pagado ---")
print("Igual a 0%:", (df["porcentaje_pagado"] == 0).sum())
print("Entre 0% y 100%:", (
    (df["porcentaje_pagado"] >= 0) &
    (df["porcentaje_pagado"] <= 100)
).sum())
print("Mayor a 100%:", (df["porcentaje_pagado"] > 100).sum())

       porcentaje_ejecutado  porcentaje_pagado
count          19632.000000       19632.000000
mean              34.731067          34.056854
std               43.151165          43.136797
min                0.000000           0.000000
25%                0.000000           0.000000
50%                0.000000           0.000000
75%               87.329793          87.169206
90%               99.369898          99.360966
95%              100.000000         100.000000
99%              100.000000         100.000000
max              424.540541         424.540541

--- Porcentaje ejecutado ---
Igual a 0%: 11310
Entre 0% y 100%: 19627
Mayor a 100%: 5

--- Porcentaje pagado ---
Igual a 0%: 11505
Entre 0% y 100%: 19627
Mayor a 100%: 5


### 2.33 Ejecución financiera según estado del contrato

Dado que los indicadores de ejecución y pago presentan una alta concentración de valores iguales a cero, se analiza su comportamiento según el estado contractual.

Esta segmentación permite diferenciar los niveles de ejecución financiera de acuerdo con la situación de los contratos y evita interpretar de manera aislada los contratos con ejecución igual a cero.

In [208]:
resumen_estado = (
    df.groupby("estado_contrato")
      .agg(
          contratos=("id_contrato", "count"),
          ejecucion_promedio=("porcentaje_ejecutado", "mean"),
          ejecucion_mediana=("porcentaje_ejecutado", "median"),
          pago_promedio=("porcentaje_pagado", "mean"),
          pago_mediano=("porcentaje_pagado", "median")
      )
      .sort_values("contratos", ascending=False)
)

resumen_estado

,contratos,ejecucion_promedio,ejecucion_mediana,pago_promedio,pago_mediano
estado_contrato,,,,,
Cerrado,9263,65.683971,88.716434,65.640521,88.682172
Modificado,5434,8.599017,0.000000,8.403874,0.000000
terminado,3416,25.412307,0.000000,25.107336,0.000000
En ejecución,2512,40.922830,48.893939,34.684939,35.833333
Borrador,2438,0.000000,0.000000,0.000000,0.000000
Aprobado,872,3.569662,0.000000,3.171977,0.000000
Cancelado,629,0.000000,0.000000,0.000000,0.000000
Suspendido,361,0.216466,0.000000,0.216466,0.000000
enviado Proveedor,301,0.000000,0.000000,0.000000,0.000000


In [209]:
terminados = df[df["estado_contrato"] == "terminado"].copy()

print("Contratos terminados:", len(terminados))

print("\nEjecución financiera:")
print("Ejecución = 0%:", (terminados["porcentaje_ejecutado"] == 0).sum())
print("Ejecución > 0%:", (terminados["porcentaje_ejecutado"] > 0).sum())

print("\nPago:")
print("Pago = 0%:", (terminados["porcentaje_pagado"] == 0).sum())
print("Pago > 0%:", (terminados["porcentaje_pagado"] > 0).sum())

Contratos terminados: 3416

Ejecución financiera:
Ejecución = 0%: 2053
Ejecución > 0%: 844

Pago:
Pago = 0%: 2061
Pago > 0%: 836


### 2.34 Distribución de los recursos pendientes

Se analiza la distribución de los valores pendientes de ejecución y de pago con el propósito de identificar la magnitud y concentración de los recursos que permanecen pendientes en los contratos.

Antes de establecer criterios de seguimiento o construir rankings, se revisan las estadísticas descriptivas y la presencia de valores extremos para comprender el comportamiento de estas variables.

In [210]:
variables_pendientes = [
    "valor_pendiente_de_ejecucion",
    "valor_pendiente_de_pago"
]

print(df[variables_pendientes].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\n--- Valores pendientes iguales a cero ---")

for variable in variables_pendientes:
    print(
        variable,
        "=", (df[variable] == 0).sum()
    )

print("\n--- Valores pendientes negativos ---")

for variable in variables_pendientes:
    print(
        variable,
        "< 0:", (df[variable] < 0).sum()
    )

       valor_pendiente_de_ejecucion  valor_pendiente_de_pago
count                  2.071800e+04             2.071800e+04
mean                   1.837707e+09             4.594587e+13
std                    1.953249e+10             6.403458e+15
min                   -4.402933e+07            -4.402933e+07
25%                    3.334536e+06             3.374416e+06
50%                    3.913187e+07             3.931667e+07
75%                    1.998125e+08             1.998671e+08
90%                    9.701495e+08             9.872179e+08
95%                    2.285150e+09             2.379131e+09
99%                    2.671183e+10             2.988059e+10
max                    6.571764e+11             9.216000e+17

--- Valores pendientes iguales a cero ---
valor_pendiente_de_ejecucion = 2486
valor_pendiente_de_pago = 2471

--- Valores pendientes negativos ---
valor_pendiente_de_ejecucion < 0: 4
valor_pendiente_de_pago < 0: 4


In [211]:
casos_pendiente_pago_extremo = df[
    df["valor_pendiente_de_pago"] > 1e12
][
    [
        "id_contrato",
        "estado_contrato",
        "tipo_de_contrato",
        "modalidad_de_contratacion",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion",
        "saldo_cdp",
        "saldo_vigencia"
    ]
]

casos_pendiente_pago_extremo

,id_contrato,estado_contrato,tipo_de_contrato,modalidad_de_contratacion,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,saldo_cdp,saldo_vigencia
6306,CO1.PCCNTR.2956499,Modificado,Concesión,Licitación pública,1.581453e+12,0.0,0.0,1.581453e+12,0.0,1.032726e+12,0.0
6743,CO1.PCCNTR.4725969,Cancelado,Prestación de servicios,Contratación directa,4.410000e+14,0.0,0.0,4.410000e+14,0.0,2.469750e+08,0.0
7850,CO1.PCCNTR.6066474,Cancelado,Prestación de servicios,Contratación directa,8.122500e+14,0.0,0.0,8.122500e+14,0.0,4.250000e+09,0.0
8445,CO1.PCCNTR.3455152,Cancelado,Prestación de servicios,Contratación directa,5.184000e+15,0.0,0.0,5.184000e+15,0.0,3.261800e+09,0.0
9109,CO1.PCCNTR.6906654,Borrador,Prestación de servicios,Contratación directa,2.859481e+14,0.0,0.0,2.859481e+14,0.0,3.140932e+09,0.0
13328,CO1.PCCNTR.736671,Cancelado,Prestación de servicios,Contratación directa,2.152960e+15,0.0,0.0,2.152960e+15,0.0,1.600000e+09,0.0
14652,CO1.PCCNTR.7506490,Cancelado,Prestación de servicios,Contratación directa,1.112709e+16,0.0,0.0,1.112709e+16,0.0,1.296091e+09,0.0
15629,CO1.PCCNTR.6303944,Cancelado,Prestación de servicios,Contratación directa,2.453221e+15,0.0,0.0,2.453221e+15,0.0,0.000000e+00,0.0
15932,CO1.PCCNTR.3247114,Cancelado,Prestación de servicios,Contratación directa,2.227840e+15,0.0,0.0,2.227840e+15,0.0,1.456866e+09,0.0
18270,CO1.PCCNTR.5233392,Cancelado,Prestación de servicios,Contratación directa,2.766550e+15,0.0,0.0,2.766550e+15,0.0,1.141267e+09,0.0


### 2.34.1 Identificación de valores financieros extremos

La revisión de la distribución de los recursos pendientes permitió identificar valores extremos en `valor_pendiente_de_pago`.

Al analizar estos registros se observa que los valores extremos corresponden a contratos con valores contractuales excepcionalmente altos y sin facturación ni pagos registrados.

Estos registros se conservan sin modificación para mantener la trazabilidad de la fuente original. Sin embargo, serán identificados como casos extremos y se evaluará su impacto sobre los análisis agregados y rankings posteriores.

In [212]:
umbrales = [1e9, 1e10, 1e11, 1e12]

print("Contratos con valor contractual superior a:")

for umbral in umbrales:
    cantidad = (
        df["valor_del_contrato"] > umbral
    ).sum()

    print(f"${umbral:,.0f}: {cantidad}")

Contratos con valor contractual superior a:
$1,000,000,000: 2111
$10,000,000,000: 412
$100,000,000,000: 99
$1,000,000,000,000: 14


In [213]:
extremos_contrato = df[
    df["valor_del_contrato"] > 1e12
][
    [
        "id_contrato",
        "estado_contrato",
        "tipo_de_contrato",
        "modalidad_de_contratacion",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion"
    ]
].sort_values(
    "valor_del_contrato",
    ascending=False
)

extremos_contrato

,id_contrato,estado_contrato,tipo_de_contrato,modalidad_de_contratacion,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion
21007,CO1.PCCNTR.2047279,Cancelado,Otro,Contratación directa,9.216000e+17,0.0,0.0,9.216000e+17,0.0
14652,CO1.PCCNTR.7506490,Cancelado,Prestación de servicios,Contratación directa,1.112709e+16,0.0,0.0,1.112709e+16,0.0
8445,CO1.PCCNTR.3455152,Cancelado,Prestación de servicios,Contratación directa,5.184000e+15,0.0,0.0,5.184000e+15,0.0
22348,CO1.PCCNTR.6589857,Cancelado,Prestación de servicios,Contratación directa,2.809000e+15,0.0,0.0,2.809000e+15,0.0
18270,CO1.PCCNTR.5233392,Cancelado,Prestación de servicios,Contratación directa,2.766550e+15,0.0,0.0,2.766550e+15,0.0
15629,CO1.PCCNTR.6303944,Cancelado,Prestación de servicios,Contratación directa,2.453221e+15,0.0,0.0,2.453221e+15,0.0
15932,CO1.PCCNTR.3247114,Cancelado,Prestación de servicios,Contratación directa,2.227840e+15,0.0,0.0,2.227840e+15,0.0
13328,CO1.PCCNTR.736671,Cancelado,Prestación de servicios,Contratación directa,2.152960e+15,0.0,0.0,2.152960e+15,0.0
7850,CO1.PCCNTR.6066474,Cancelado,Prestación de servicios,Contratación directa,8.122500e+14,0.0,0.0,8.122500e+14,0.0
6743,CO1.PCCNTR.4725969,Cancelado,Prestación de servicios,Contratación directa,4.410000e+14,0.0,0.0,4.410000e+14,0.0


In [214]:
total_pendiente_pago = df["valor_pendiente_de_pago"].sum()
total_pendiente_ejecucion = df["valor_pendiente_de_ejecucion"].sum()

extremos = df["valor_del_contrato"] > 1e12

pendiente_pago_extremos = (
    df.loc[extremos, "valor_pendiente_de_pago"].sum()
)

pendiente_ejecucion_extremos = (
    df.loc[extremos, "valor_pendiente_de_ejecucion"].sum()
)

print("Pendiente de pago total:", total_pendiente_pago)
print("Pendiente de pago en extremos:", pendiente_pago_extremos)
print(
    "Participación extremos:",
    pendiente_pago_extremos / total_pendiente_pago * 100
)

print("\nPendiente de ejecución total:", total_pendiente_ejecucion)
print("Pendiente de ejecución en extremos:", pendiente_ejecucion_extremos)
print(
    "Participación extremos:",
    pendiente_ejecucion_extremos / total_pendiente_ejecucion * 100
)

Pendiente de pago total: 9.519065759384841e+17
Pendiente de pago en extremos: 9.518675019432721e+17
Participación extremos: 99.99589518591428

Pendiente de ejecución total: 38073619246265.0
Pendiente de ejecución en extremos: 0.0
Participación extremos: 0.0


### 2.35 Recursos pendientes de ejecución

La variable `valor_pendiente_de_ejecucion` permite identificar los recursos contractuales que, de acuerdo con la información financiera disponible, permanecen pendientes de ejecución.

Antes de construir rankings de contratos, se analiza la distribución de esta variable para determinar su comportamiento, concentración y presencia de valores extremos.

Los valores originales del dataset se conservan sin modificación. Los casos negativos identificados previamente se consideran inconsistencias financieras y serán tratados de manera explícita en los análisis posteriores.

In [215]:
pendiente_ejecucion_positivo = df[
    df["valor_pendiente_de_ejecucion"] > 0
].copy()

print("Contratos con pendiente de ejecución > 0:",
      len(pendiente_ejecucion_positivo))

print("\nValor total pendiente de ejecución:",
      pendiente_ejecucion_positivo[
          "valor_pendiente_de_ejecucion"
      ].sum())

print("\nValor promedio:",
      pendiente_ejecucion_positivo[
          "valor_pendiente_de_ejecucion"
      ].mean())

print("\nValor mediano:",
      pendiente_ejecucion_positivo[
          "valor_pendiente_de_ejecucion"
      ].median())

Contratos con pendiente de ejecución > 0: 18228

Valor total pendiente de ejecución: 38073669948817.0

Valor promedio: 2088746431.249561

Valor mediano: 56000004.0


### Resultado

Se identificaron 18.228 contratos con valores positivos de recursos pendientes de ejecución, por un valor agregado aproximado de $38,07 billones.

La mediana del valor pendiente es de $56,0 millones, mientras que el promedio alcanza $2.088,7 millones. Esta diferencia evidencia la presencia de contratos con valores pendientes significativamente superiores al comportamiento típico de la distribución.

Estos resultados servirán como base para identificar los contratos con mayores recursos pendientes de ejecución y priorizar aquellos que puedan requerir seguimiento.

In [216]:
top_pendiente_ejecucion = (
    df[df["valor_pendiente_de_ejecucion"] > 0]
    [
        [
            "id_contrato",
            "estado_contrato",
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "valor_del_contrato",
            "valor_facturado",
            "valor_pagado",
            "valor_pendiente_de_pago",
            "valor_pendiente_de_ejecucion",
            "dias_adicionados"
        ]
    ]
    .sort_values(
        "valor_pendiente_de_ejecucion",
        ascending=False
    )
    .head(15)
)

top_pendiente_ejecucion

,id_contrato,estado_contrato,tipo_de_contrato,modalidad_de_contratacion,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,dias_adicionados
15978,CO1.PCCNTR.3633179,Modificado,Obra,Licitación pública Obra Publica,6.571764e+11,0.0,0.0,6.571764e+11,6.571764e+11,0.0
9078,CO1.PCCNTR.9496559,Aprobado,Obra,Licitación pública Obra Publica,6.503996e+11,0.0,0.0,6.503996e+11,6.503996e+11,0.0
16191,CO1.PCCNTR.2498523,Modificado,Obra,Licitación pública Obra Publica,6.104614e+11,0.0,0.0,6.104614e+11,6.104614e+11,0.0
3513,CO1.PCCNTR.2410198,Modificado,Obra,Licitación pública Obra Publica,5.933837e+11,0.0,0.0,5.933837e+11,5.933837e+11,0.0
16674,CO1.PCCNTR.2498450,Modificado,Obra,Licitación pública Obra Publica,5.413900e+11,0.0,0.0,5.413900e+11,5.413900e+11,0.0
25138,CO1.PCCNTR.9376809,Aprobado,Obra,Licitación pública Obra Publica,5.117431e+11,0.0,0.0,5.117431e+11,5.117431e+11,0.0
8065,CO1.PCCNTR.3035011,Modificado,Otro,Contratación directa,5.096140e+11,0.0,0.0,5.096140e+11,5.096140e+11,0.0
2762,CO1.PCCNTR.3155147,Modificado,Obra,Licitación pública Obra Publica,4.963449e+11,0.0,0.0,4.963449e+11,4.963449e+11,1096.0
5004,CO1.PCCNTR.3144306,Modificado,Obra,Licitación pública Obra Publica,4.547759e+11,0.0,0.0,4.547759e+11,4.547759e+11,0.0
8616,CO1.PCCNTR.2410088,Modificado,Obra,Licitación pública Obra Publica,4.338918e+11,0.0,0.0,4.338918e+11,4.338918e+11,0.0


### 2.36 Mayores recursos pendientes de ejecución

Se identificaron los contratos con mayores valores pendientes de ejecución con el propósito de establecer cuáles concentran los mayores recursos aún no ejecutados.

Los contratos con mayores saldos corresponden principalmente a contratos de obra y presentan, en su mayoría, modalidad de Licitación pública Obra Pública y estado contractual Modificado.

El ranking permite identificar los contratos con mayor exposición financiera pendiente. Sin embargo, un valor elevado pendiente de ejecución no constituye por sí solo una evidencia de irregularidad, por lo que posteriormente se contrastará con variables asociadas a modificaciones y ampliaciones del plazo contractual.

### 2.37 Extensiones del plazo contractual

Se identificaron 3.728 contratos con días adicionados al plazo contractual. La mediana de ampliación es de 61 días, mientras que el promedio alcanza 107 días, evidenciando la existencia de contratos con ampliaciones considerablemente superiores al comportamiento típico.

Los mayores valores de días adicionados se concentran principalmente en contratos con estado Modificado. Esta variable se utilizará como una señal complementaria para identificar contratos que puedan requerir seguimiento, junto con los recursos pendientes de ejecución.

In [217]:
print("Contratos con días adicionados > 0:",
      (df["dias_adicionados"] > 0).sum())

print("\nDías adicionados:")
print(
    df.loc[df["dias_adicionados"] > 0, "dias_adicionados"]
    .describe()
)

print("\nMáximos días adicionados:")

print(
    df[
        df["dias_adicionados"] > 0
    ][
        [
            "id_contrato",
            "estado_contrato",
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "valor_del_contrato",
            "dias_adicionados"
        ]
    ]
    .sort_values(
        "dias_adicionados",
        ascending=False
    )
    .head(15)
)

Contratos con días adicionados > 0: 3728

Días adicionados:
count    3728.000000
mean      107.056330
std       147.632493
min         1.000000
25%        30.000000
50%        61.000000
75%       121.000000
max      3287.000000
Name: dias_adicionados, dtype: float64

Máximos días adicionados:
              id_contrato estado_contrato tipo_de_contrato  \
15193  CO1.PCCNTR.8127893      Modificado             Otro   
3794   CO1.PCCNTR.7067617      Modificado             Otro   
5598    CO1.PCCNTR.622805      Modificado         Comodato   
15023  CO1.PCCNTR.2355024      Modificado         Comodato   
19791  CO1.PCCNTR.1655852      Modificado         Comodato   
8136    CO1.PCCNTR.784382      Modificado         Comodato   
646    CO1.PCCNTR.8104041      Modificado             Otro   
16552  CO1.PCCNTR.6461939      Modificado             Otro   
4885    CO1.PCCNTR.630456      Modificado         Comodato   
24599  CO1.PCCNTR.4313818       terminado         Comodato   
7      CO1.PCCNTR.819794

### 2.38 Cruce de recursos pendientes de ejecución y ampliación del plazo

Una vez analizadas de manera independiente las variables `valor_pendiente_de_ejecucion` y `dias_adicionados`, se realiza un cruce entre ambas para identificar contratos que presenten simultáneamente un nivel elevado de recursos pendientes de ejecución y una ampliación significativa del plazo contractual.

Para evitar establecer umbrales arbitrarios, se utilizará el percentil 90 de cada variable como referencia. Se considerarán como candidatos prioritarios aquellos contratos que superen simultáneamente ambos valores de referencia.

Este análisis no busca determinar la existencia de irregularidades, sino generar una señal de priorización para el seguimiento de los contratos que combinan una mayor exposición financiera pendiente con extensiones relevantes del plazo contractual.

In [218]:
p90_pendiente = df["valor_pendiente_de_ejecucion"].quantile(0.90)
p90_dias = df["dias_adicionados"].quantile(0.90)

print("P90 pendiente de ejecución:", p90_pendiente)
print("P90 días adicionados:", p90_dias)

seguimiento = df[
    (df["valor_pendiente_de_ejecucion"] > p90_pendiente) &
    (df["dias_adicionados"] > p90_dias)
].copy()

print("\nContratos que superan ambos P90:", len(seguimiento))

P90 pendiente de ejecución: 970149547.8999996
P90 días adicionados: 58.0

Contratos que superan ambos P90: 750


In [219]:
top_seguimiento = (
    seguimiento[
        [
            "id_contrato",
            "estado_contrato",
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "valor_del_contrato",
            "valor_facturado",
            "valor_pagado",
            "valor_pendiente_de_pago",
            "valor_pendiente_de_ejecucion",
            "dias_adicionados"
        ]
    ]
    .sort_values(
        ["valor_pendiente_de_ejecucion", "dias_adicionados"],
        ascending=[False, False]
    )
    .head(15)
)

top_seguimiento

,id_contrato,estado_contrato,tipo_de_contrato,modalidad_de_contratacion,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,dias_adicionados
2762,CO1.PCCNTR.3155147,Modificado,Obra,Licitación pública Obra Publica,4.963449e+11,0.0,0.0,4.963449e+11,4.963449e+11,1096.0
13407,CO1.PCCNTR.335410,Modificado,Otro,Contratación Directa (con ofertas),3.150360e+11,0.0,0.0,3.150360e+11,3.150360e+11,365.0
2753,CO1.PCCNTR.6917458,Modificado,Obra,Licitación pública Obra Publica,2.212971e+11,0.0,0.0,2.212971e+11,2.212971e+11,153.0
6171,CO1.PCCNTR.7014395,Modificado,Obra,Licitación pública Obra Publica,2.081305e+11,0.0,0.0,2.081305e+11,2.081305e+11,153.0
8672,CO1.PCCNTR.2086648,Modificado,Obra,Licitación pública Obra Publica,2.052199e+11,0.0,0.0,2.052199e+11,2.052199e+11,396.0
3616,CO1.PCCNTR.6971120,Modificado,Obra,Licitación pública Obra Publica,1.541993e+11,0.0,0.0,1.541993e+11,1.541993e+11,153.0
11765,CO1.PCCNTR.2529861,Modificado,Otro,Contratación directa,8.187543e+10,0.0,0.0,8.187543e+10,8.187543e+10,364.0
19735,CO1.PCCNTR.2543943,Modificado,Otro,Contratación directa,8.035000e+10,0.0,0.0,8.035000e+10,8.035000e+10,212.0
16956,CO1.PCCNTR.5136002,Modificado,Otro,Contratación directa,7.817092e+10,0.0,0.0,7.817092e+10,7.817092e+10,274.0
10567,CO1.PCCNTR.2052307,Modificado,Otro,Contratación directa,7.576161e+10,0.0,0.0,7.576161e+10,7.576161e+10,365.0


### Resultado del cruce

El cruce de las variables de recursos pendientes de ejecución y días adicionados permitió identificar 750 contratos que superan simultáneamente los percentiles 90 de ambas variables.

En los contratos con mayores valores pendientes se observa una presencia predominante del estado Modificado, acompañada en varios casos por ampliaciones significativas del plazo contractual y ausencia de facturación y pagos registrados.

Estos resultados permiten construir una lista priorizada de contratos para seguimiento, entendiendo que las señales identificadas no constituyen por sí mismas evidencia de irregularidad.

### 2.39 Posibilidad de prórroga contractual

Se revisa la variable `el_contrato_puede_ser_prorrogado` para determinar su distribución y establecer si aporta información adicional al análisis de las extensiones del plazo contractual.

Esta variable se utilizará únicamente como contexto complementario, dado que los días adicionados ya permiten medir directamente la extensión efectiva del plazo.

In [220]:
print(df["el_contrato_puede_ser_prorrogado"].value_counts(dropna=False))

el_contrato_puede_ser_prorrogado
No     17173
NaN     4906
Si      3526
Name: count, dtype: int64


### Resultado

La variable `el_contrato_puede_ser_prorrogado` registra 3.526 contratos con posibilidad de prórroga, 17.173 contratos que indican que no pueden ser prorrogados y 4.906 registros sin información.

Dado que `dias_adicionados` permite medir directamente las extensiones efectivamente registradas, la posibilidad de prórroga se utilizará como variable contextual y no como criterio adicional para el ranking de seguimiento.

## 2.40. Caracterización de los contratos prioritarios

A partir de los criterios definidos previamente, se identificaron los contratos que presentan simultáneamente un valor pendiente de ejecución superior al percentil 90 (P90) y un número de días adicionados superior al P90.

En este bloque se caracteriza este grupo prioritario a partir de su estado contractual, tipo de contrato, modalidad de contratación, valor contractual, recursos pendientes de ejecución y días adicionados.

El objetivo no es establecer que estos contratos presenten irregularidades, sino identificar un conjunto de contratos que, por la combinación de estas señales, puede requerir un mayor nivel de seguimiento dentro del análisis.

## Definición de los umbrales de priorización

Para identificar contratos que puedan requerir seguimiento se utilizan dos umbrales definidos mediante el percentil 90 (P90):

- **P90 del valor pendiente de ejecución:** $970.149.547,90.
- **P90 de días adicionados:** 58 días.

Estos umbrales permiten identificar contratos que presentan simultáneamente niveles elevados de recursos pendientes de ejecución y extensiones del plazo contractual respecto al comportamiento observado en la mayoría de los contratos analizados.

In [222]:
# Definición de los umbrales P90 utilizados para la priorización

p90_pendiente_ejecucion = df_financiero[
    "valor_pendiente_de_ejecucion"
].quantile(0.90)

p90_dias_adicionados = df_financiero[
    "dias_adicionados"
].quantile(0.90)

print("P90 pendiente de ejecución:", p90_pendiente_ejecucion)
print("P90 días adicionados:", p90_dias_adicionados)

P90 pendiente de ejecución: 970149547.8999996
P90 días adicionados: 58.0


In [224]:
contratos_prioritarios = df_financiero[
    (df_financiero["valor_pendiente_de_ejecucion"] > p90_pendiente_ejecucion) &
    (df_financiero["dias_adicionados"] > p90_dias_adicionados)
].copy()

print("Contratos prioritarios:", len(contratos_prioritarios))

Contratos prioritarios: 750


## 2.41. Caracterización de los contratos prioritarios

Los 750 contratos identificados mediante los criterios de priorización serán caracterizados según su estado contractual, tipo de contrato y modalidad de contratación.

Esta caracterización permite identificar cómo se distribuye el grupo prioritario y establecer los principales patrones que posteriormente podrán ser incorporados al análisis y al tablero de control.

El objetivo es describir el grupo identificado, sin interpretar estas características como evidencia de irregularidades.

In [225]:
prioritarios_estado = (
    contratos_prioritarios["estado_contrato"]
    .value_counts()
    .rename("contratos")
    .to_frame()
)

prioritarios_estado

,contratos
estado_contrato,
Modificado,689
Suspendido,57
En ejecución,3
terminado,1


In [226]:
prioritarios_tipo = (
    contratos_prioritarios["tipo_de_contrato"]
    .value_counts()
    .rename("contratos")
    .to_frame()
)

prioritarios_tipo

,contratos
tipo_de_contrato,
Obra,274
Otro,154
Prestación de servicios,134
Interventoría,108
Consultoría,75
Compraventa,4
Suministros,1


In [227]:
prioritarios_modalidad = (
    contratos_prioritarios["modalidad_de_contratacion"]
    .value_counts()
    .rename("contratos")
    .to_frame()
)

prioritarios_modalidad

,contratos
modalidad_de_contratacion,
Contratación directa,322
Concurso de méritos abierto,183
Licitación pública,119
Licitación pública Obra Publica,89
Selección Abreviada de Menor Cuantía,18
Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes,10
Selección abreviada subasta inversa,7
Contratación Directa (con ofertas),2


### Interpretación

Los contratos prioritarios presentan una alta concentración tanto por estado contractual como por tipo y modalidad de contratación. El 99,5% se encuentra en estado modificado o suspendido, mientras que el 99,3% corresponde a los cinco principales tipos de contrato identificados.

Por modalidad de contratación, las cuatro categorías principales concentran el 95,1% de los contratos prioritarios, destacándose la contratación directa con 322 contratos (42,9%).

Estos resultados permiten caracterizar el grupo priorizado y orientar su análisis posterior, sin considerar estas concentraciones como evidencia de irregularidades. La identificación de los contratos se fundamenta exclusivamente en la combinación de valores elevados de recursos pendientes de ejecución y días adicionados.

## Cierre de la selección y limpieza de datos

En esta etapa se realizó la selección, revisión y limpieza de las variables utilizadas para el análisis de la Pregunta 3. El proceso incluyó la revisión de la estructura del conjunto de datos, identificación de variables sin variabilidad, análisis de valores faltantes, revisión de variables categóricas y evaluación de la consistencia de las variables financieras.

Se identificaron y documentaron registros con valores contractuales iguales a cero, valores financieros negativos y valores extremos, así como posibles inconsistencias entre el valor contractual, los valores facturados y pagados y los valores pendientes. Estas situaciones no fueron eliminadas automáticamente, dado que pueden corresponder a características propias de los registros de contratación y deben ser consideradas durante las etapas posteriores del análisis.

También se generaron indicadores financieros derivados, como el porcentaje ejecutado y el porcentaje pagado, y se identificaron señales asociadas al valor pendiente de ejecución y a los días adicionados al contrato.

Como resultado, se obtiene un conjunto de datos revisado y documentado que constituye la base para la siguiente etapa. Las transformaciones y variables específicas requeridas para el análisis serán desarrolladas en `03_alistamiento_p3.ipynb`, evitando incorporar análisis exploratorio o criterios de visualización dentro de esta etapa de limpieza.

**Resultado de la etapa:** dataset seleccionado, revisado y documentado, listo para el alistamiento y posterior exploración de los datos.